# Task 11 — MCI Benchmarking and Sensitivity Analysis

This notebook reconstructs the locked Molecular Concordance Index and
evaluates it against simpler benchmark methods.

It also performs:

- component-weight sensitivity analysis;
- threshold-grid sensitivity analysis;
- leave-one-component-out stress tests;
- leave-one-HCM-cohort-out analysis;
- held-out expression benchmarking;
- GWAS Catalog convergence benchmarking;
- publication-ready table and figure generation.

## Execution

The setup cell reuses the current repository when available. When the
repository is absent, it clones the public GitHub repository into the
current Colab environment. It never deletes an existing repository.

All Task 11 outputs are written beneath:

`results/task11_benchmarking/`


In [ ]:
# STEP 1 — Locate or clone the repository and inspect required inputs

import os
import subprocess
from pathlib import Path

REPO_URL = (
    "https://github.com/SANGHATI23/"
    "mci-cardiomyopathy-concordance.git"
)

REPO_NAME = "mci-cardiomyopathy-concordance"


def is_project_repository(path):
    """Return True when path appears to be the MCI Git repository."""
    path = Path(path)

    return (
        (path / ".git").is_dir()
        and (path / "results" / "mci_scores").is_dir()
    )


# Search the current directory and its parents first. This supports
# running the notebook from an existing local clone or from Jupyter.
search_candidates = [
    Path.cwd(),
    *Path.cwd().parents,
]

# Standard Colab clone location.
colab_target = Path("/content") / REPO_NAME
search_candidates.append(colab_target)

PROJECT_DIR = next(
    (
        candidate.resolve()
        for candidate in search_candidates
        if is_project_repository(candidate)
    ),
    None,
)

if PROJECT_DIR is None:
    if colab_target.exists():
        raise RuntimeError(
            "The expected Colab target already exists but is not a "
            "recognized Git repository. Rename or remove that directory "
            "manually before rerunning this notebook."
        )

    subprocess.run(
        [
            "git",
            "clone",
            REPO_URL,
            str(colab_target),
        ],
        check=True,
    )

    PROJECT_DIR = colab_target.resolve()

os.chdir(PROJECT_DIR)

print("=" * 80)
print("CURRENT PROJECT DIRECTORY")
print("=" * 80)
print(PROJECT_DIR)

print("\n" + "=" * 80)
print("CURRENT GIT BRANCH AND STATUS")
print("=" * 80)

subprocess.run(
    ["git", "branch", "--show-current"],
    check=True,
)

subprocess.run(
    ["git", "status", "--short"],
    check=True,
)

print("\n" + "=" * 80)
print("KEY HCM MCI INPUT FILES")
print("=" * 80)

MCI_DIR = PROJECT_DIR / "results" / "mci_scores"

expected_files = [
    "TASK4_CODE23_HCM_only_clean_DE_input_used_for_MCI.csv",
    "TASK4_CODE23_REAL_HCM_MCI_scores_primary.csv",
    "TASK4_CODE23_REAL_HCM_MCI_scores_primary_wide_per_cohort.csv",
    "TASK6_FINAL_HCM_MCI_WITH_GTEx_AND_BOOTSTRAP_CI.csv",
    "TASK9_PATHA_HELDOUT_GSE160997_VALIDATION_GENE_LEVEL.csv",
    "TASK9_PATHB_GWAS_CATALOG_CONVERGENCE_GENE_LEVEL.csv",
]

missing_files = []

for filename in expected_files:
    path = MCI_DIR / filename
    exists = path.exists()

    if not exists:
        missing_files.append(filename)

    status = "FOUND" if exists else "MISSING"
    size = (
        f"{path.stat().st_size:,} bytes"
        if exists
        else "—"
    )

    print(
        f"{status:7} | "
        f"{size:>14} | "
        f"{filename}"
    )

if missing_files:
    raise FileNotFoundError(
        "Required analysis inputs are missing: "
        + ", ".join(missing_files)
    )

print("\nSTEP 1 COMPLETE")


In [ ]:
# STEP 2 — Inspect schemas and identify fields needed for benchmarking

import pandas as pd
from pathlib import Path

mci_dir = PROJECT_DIR / "results" / "mci_scores"

file_paths = {
    "clean_de": mci_dir / "TASK4_CODE23_HCM_only_clean_DE_input_used_for_MCI.csv",
    "primary_mci": mci_dir / "TASK4_CODE23_REAL_HCM_MCI_scores_primary.csv",
    "wide_mci": mci_dir / "TASK4_CODE23_REAL_HCM_MCI_scores_primary_wide_per_cohort.csv",
    "final_resource": mci_dir / "TASK6_FINAL_HCM_MCI_WITH_GTEx_AND_BOOTSTRAP_CI.csv",
    "heldout": mci_dir / "TASK9_PATHA_HELDOUT_GSE160997_VALIDATION_GENE_LEVEL.csv",
    "gwas": mci_dir / "TASK9_PATHB_GWAS_CATALOG_CONVERGENCE_GENE_LEVEL.csv",
}

dfs = {}

for name, path in file_paths.items():
    dfs[name] = pd.read_csv(path)

    print("\n" + "=" * 100)
    print(f"{name.upper()} | FILE: {path.name}")
    print("=" * 100)
    print(f"Shape: {dfs[name].shape}")
    print("\nColumns:")

    for i, column in enumerate(dfs[name].columns, start=1):
        print(f"{i:>2}. {column}")

# Focused inspection of the clean cohort-level DE input
clean_de = dfs["clean_de"]

print("\n" + "=" * 100)
print("CLEAN DE INPUT — FIRST 8 ROWS")
print("=" * 100)
print(clean_de.head(8).to_string(index=False))

print("\n" + "=" * 100)
print("CLEAN DE INPUT — DATA TYPES")
print("=" * 100)
print(clean_de.dtypes.to_string())

print("\n" + "=" * 100)
print("CLEAN DE INPUT — MISSING VALUES")
print("=" * 100)
missing = clean_de.isna().sum()
print(missing[missing > 0].sort_values(ascending=False).to_string())

# Automatically identify likely benchmarking columns
keywords = {
    "gene": ["gene", "symbol"],
    "cohort": ["cohort", "dataset", "study", "gse"],
    "log2fc": ["log2fc", "logfc", "fold_change", "foldchange"],
    "standard_error": ["se", "stderr", "standard_error", "lfcse"],
    "p_value": ["pvalue", "p_value", "pval"],
    "fdr": ["fdr", "padj", "adj_p", "adjusted_p"],
}

print("\n" + "=" * 100)
print("LIKELY BENCHMARKING COLUMNS")
print("=" * 100)

lower_columns = {col: col.lower() for col in clean_de.columns}

for field, terms in keywords.items():
    matches = [
        original
        for original, lower in lower_columns.items()
        if any(term in lower for term in terms)
    ]
    print(f"{field:>16}: {matches}")

# Show cohort/dataset values when such a column can be detected
cohort_candidates = [
    col for col in clean_de.columns
    if any(term in col.lower() for term in ["cohort", "dataset", "study", "gse"])
]

if cohort_candidates:
    cohort_col = cohort_candidates[0]
    print("\n" + "=" * 100)
    print(f"UNIQUE VALUES IN POSSIBLE COHORT COLUMN: {cohort_col}")
    print("=" * 100)
    print(clean_de[cohort_col].value_counts(dropna=False).to_string())
else:
    print("\nNo likely cohort column was automatically identified.")

print("\nSTEP 2 COMPLETE")


In [ ]:
# STEP 3 — Independently reconstruct the published MCI from cohort-level data

import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations

PROJECT_DIR = PROJECT_DIR
MCI_DIR = PROJECT_DIR / "results" / "mci_scores"

clean_path = MCI_DIR / "TASK4_CODE23_HCM_only_clean_DE_input_used_for_MCI.csv"
primary_path = MCI_DIR / "TASK4_CODE23_REAL_HCM_MCI_scores_primary.csv"

clean_de = pd.read_csv(clean_path)
primary_mci = pd.read_csv(primary_path)

# Keep only the rows that were eligible for the original MCI calculation
eligible = clean_de.loc[
    (clean_de["score_eligible"] == True)
    & (clean_de["matched_in_DE"] == True)
].copy()

# Convert essential fields explicitly to numeric
for col in ["log2FC", "SE", "p_value", "FDR"]:
    eligible[col] = pd.to_numeric(eligible[col], errors="coerce")

print("=" * 100)
print("INPUT INTEGRITY CHECK")
print("=" * 100)

print(f"Eligible cohort-level rows: {len(eligible)}")
print(f"Unique genes: {eligible['gene_symbol'].nunique()}")
print(f"Unique cohorts: {eligible['cohort'].nunique()}")
print(f"Cohorts: {sorted(eligible['cohort'].dropna().unique().tolist())}")

duplicate_count = eligible.duplicated(
    subset=["gene_symbol", "cohort"],
    keep=False
).sum()

print(f"Duplicate gene-cohort rows: {duplicate_count}")
print(f"Missing log2FC: {eligible['log2FC'].isna().sum()}")
print(f"Missing SE: {eligible['SE'].isna().sum()}")
print(f"Missing p-value: {eligible['p_value'].isna().sum()}")
print(f"Missing FDR: {eligible['FDR'].isna().sum()}")
print(f"Non-positive SE values: {(eligible['SE'] <= 0).sum()}")


def calculate_direction_agreement(values):
    """
    Proportion of all cohort pairs that have the same non-zero log2FC sign.
    """
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) < 2:
        return np.nan

    signs = np.sign(values)
    pairs = list(combinations(signs, 2))

    if len(pairs) == 0:
        return np.nan

    agreements = [int(a == b) for a, b in pairs]
    return float(np.mean(agreements))


def calculate_effect_consistency(values):
    """
    S_g = 1 - min(CV / 2, 1)
    CV = sample SD(log2FC) / abs(mean(log2FC))
    """
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) < 2:
        return np.nan

    mean_value = np.mean(values)
    sd_value = np.std(values, ddof=1)

    if np.isclose(mean_value, 0.0):
        return 0.0

    cv = sd_value / abs(mean_value)
    return float(1.0 - min(cv / 2.0, 1.0))


def assign_mci_tier(score, n_cohorts):
    if n_cohorts < 2 or pd.isna(score):
        return "INSUFFICIENT_COVERAGE"
    if score >= 0.70:
        return "HIGH"
    if score >= 0.45:
        return "MODERATE"
    return "UNSTABLE"


reconstructed_rows = []

for gene, group in eligible.groupby("gene_symbol", sort=True):
    log2fc = group["log2FC"].dropna().to_numpy(dtype=float)
    fdr = group["FDR"].dropna().to_numpy(dtype=float)

    n_cohorts = len(log2fc)
    mean_log2fc = np.mean(log2fc) if n_cohorts > 0 else np.nan
    sd_log2fc = np.std(log2fc, ddof=1) if n_cohorts >= 2 else np.nan

    D_g = calculate_direction_agreement(log2fc)
    S_g = calculate_effect_consistency(log2fc)
    R_g = np.mean(fdr < 0.05) if len(fdr) > 0 else np.nan

    if n_cohorts >= 2 and all(pd.notna(x) for x in [D_g, S_g, R_g]):
        reconstructed_mci = (0.40 * D_g) + (0.35 * S_g) + (0.25 * R_g)
    else:
        reconstructed_mci = np.nan

    reconstructed_rows.append(
        {
            "gene_symbol": gene,
            "reconstructed_n_cohorts": n_cohorts,
            "reconstructed_mean_log2FC": mean_log2fc,
            "reconstructed_sd_log2FC": sd_log2fc,
            "reconstructed_D_g": D_g,
            "reconstructed_S_g": S_g,
            "reconstructed_R_g": R_g,
            "reconstructed_MCI": reconstructed_mci,
            "reconstructed_tier": assign_mci_tier(
                reconstructed_mci,
                n_cohorts
            ),
        }
    )

reconstructed = pd.DataFrame(reconstructed_rows)

audit = primary_mci.merge(
    reconstructed,
    on="gene_symbol",
    how="outer",
    validate="one_to_one"
)

# Numerical differences
comparison_pairs = {
    "n_cohorts": (
        "n_hcm_cohorts_available",
        "reconstructed_n_cohorts"
    ),
    "mean_log2FC": (
        "mean_log2FC",
        "reconstructed_mean_log2FC"
    ),
    "sd_log2FC": (
        "sd_log2FC",
        "reconstructed_sd_log2FC"
    ),
    "D_g": (
        "D_g_direction_agreement",
        "reconstructed_D_g"
    ),
    "S_g": (
        "S_g_effect_size_consistency",
        "reconstructed_S_g"
    ),
    "R_g": (
        "R_g_statistical_reproducibility",
        "reconstructed_R_g"
    ),
    "MCI": (
        "MCI",
        "reconstructed_MCI"
    ),
}

print("\n" + "=" * 100)
print("RECONSTRUCTION AGREEMENT")
print("=" * 100)

for label, (original_col, reconstructed_col) in comparison_pairs.items():
    valid = audit[[original_col, reconstructed_col]].dropna()

    if len(valid) == 0:
        print(f"{label:>12}: no comparable values")
        continue

    differences = (
        valid[original_col].astype(float)
        - valid[reconstructed_col].astype(float)
    ).abs()

    print(
        f"{label:>12}: "
        f"n={len(valid):>2}, "
        f"max absolute difference={differences.max():.12g}, "
        f"differences > 1e-10={(differences > 1e-10).sum()}"
    )

# Normalize tier labels before comparison
def normalize_tier(value):
    if pd.isna(value):
        return "MISSING"

    value = str(value).strip().upper()
    value = value.replace(" ", "_").replace("-", "_")

    if "INSUFFICIENT" in value:
        return "INSUFFICIENT_COVERAGE"

    return value


audit["original_tier_normalized"] = audit["MCI_tier"].apply(normalize_tier)
audit["reconstructed_tier_normalized"] = audit["reconstructed_tier"].apply(
    normalize_tier
)

tier_mismatches = audit.loc[
    audit["original_tier_normalized"]
    != audit["reconstructed_tier_normalized"],
    [
        "gene_symbol",
        "n_hcm_cohorts_available",
        "MCI",
        "MCI_tier",
        "reconstructed_MCI",
        "reconstructed_tier",
    ],
]

print("\n" + "=" * 100)
print("TIER AGREEMENT")
print("=" * 100)
print(f"Total genes compared: {len(audit)}")
print(f"Tier mismatches: {len(tier_mismatches)}")

if len(tier_mismatches) > 0:
    print("\nTier mismatch details:")
    print(tier_mismatches.to_string(index=False))
else:
    print("All reconstructed tiers match the original tiers.")

# Show any important MCI discrepancies
mci_discrepancies = audit.loc[
    (
        audit["MCI"].notna()
        & audit["reconstructed_MCI"].notna()
        & (
            audit["MCI"] - audit["reconstructed_MCI"]
        ).abs().gt(1e-10)
    ),
    [
        "gene_symbol",
        "D_g_direction_agreement",
        "reconstructed_D_g",
        "S_g_effect_size_consistency",
        "reconstructed_S_g",
        "R_g_statistical_reproducibility",
        "reconstructed_R_g",
        "MCI",
        "reconstructed_MCI",
    ],
]

print("\n" + "=" * 100)
print("MCI DISCREPANCY DETAILS")
print("=" * 100)

if len(mci_discrepancies) == 0:
    print("No MCI discrepancies greater than 1e-10.")
else:
    print(mci_discrepancies.to_string(index=False))

print("\nSTEP 3 COMPLETE")


In [ ]:
# STEP 4 — Construct gene-level alternative benchmark methods
#
# Methods:
# 1. Original weighted MCI
# 2. Equal-weight MCI composite
# 3. Majority direction-vote score
# 4. Fisher combined significance
# 5. Direction-aware Stouffer combined significance
# 6. DerSimonian-Laird random-effects meta-analysis

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import chi2, norm

warnings.filterwarnings("ignore")

PROJECT_DIR = PROJECT_DIR
MCI_DIR = PROJECT_DIR / "results" / "mci_scores"
OUTPUT_DIR = PROJECT_DIR / "results" / "task11_benchmarking"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

clean_path = MCI_DIR / "TASK4_CODE23_HCM_only_clean_DE_input_used_for_MCI.csv"
primary_path = MCI_DIR / "TASK4_CODE23_REAL_HCM_MCI_scores_primary.csv"

clean_de = pd.read_csv(clean_path)
primary_mci = pd.read_csv(primary_path)

eligible = clean_de.loc[
    (clean_de["score_eligible"] == True)
    & (clean_de["matched_in_DE"] == True)
].copy()

for column in ["log2FC", "SE", "p_value", "FDR"]:
    eligible[column] = pd.to_numeric(
        eligible[column],
        errors="coerce"
    )

P_FLOOR = 1e-300


def safe_minus_log10(p_value):
    if pd.isna(p_value):
        return np.nan

    return float(
        -np.log10(
            max(float(p_value), P_FLOOR)
        )
    )


def fisher_combination(p_values):
    """
    Fisher's combined probability test.

    This method is significance-based and does not use effect direction.
    """
    p_values = np.asarray(p_values, dtype=float)
    p_values = p_values[np.isfinite(p_values)]

    if len(p_values) < 2:
        return np.nan, np.nan

    p_values = np.clip(p_values, P_FLOOR, 1.0)

    statistic = -2.0 * np.sum(np.log(p_values))
    degrees_freedom = 2 * len(p_values)
    combined_p = chi2.sf(statistic, degrees_freedom)

    return float(statistic), float(combined_p)


def signed_stouffer_combination(p_values, effects):
    """
    Direction-aware, equal-weight Stouffer method.

    Each two-sided cohort p-value is converted to an absolute Z score,
    after which the sign of log2FC is restored.
    """
    p_values = np.asarray(p_values, dtype=float)
    effects = np.asarray(effects, dtype=float)

    valid = (
        np.isfinite(p_values)
        & np.isfinite(effects)
    )

    p_values = p_values[valid]
    effects = effects[valid]

    if len(p_values) < 2:
        return np.nan, np.nan

    p_values = np.clip(p_values, P_FLOOR, 1.0)

    absolute_z = norm.isf(p_values / 2.0)
    signed_z = np.sign(effects) * absolute_z

    combined_z = np.sum(signed_z) / np.sqrt(len(signed_z))
    combined_p = 2.0 * norm.sf(abs(combined_z))

    return float(combined_z), float(combined_p)


def dersimonian_laird_random_effects(effects, standard_errors):
    """
    DerSimonian-Laird random-effects meta-analysis.

    Returns:
    - pooled random-effects log2FC
    - pooled standard error
    - Z statistic
    - two-sided p-value
    - tau-squared
    - I-squared
    - Cochran Q
    """
    effects = np.asarray(effects, dtype=float)
    standard_errors = np.asarray(standard_errors, dtype=float)

    valid = (
        np.isfinite(effects)
        & np.isfinite(standard_errors)
        & (standard_errors > 0)
    )

    effects = effects[valid]
    standard_errors = standard_errors[valid]

    k = len(effects)

    if k < 2:
        return {
            "random_effect_log2FC": np.nan,
            "random_effect_SE": np.nan,
            "random_effect_z": np.nan,
            "random_effect_p": np.nan,
            "tau2": np.nan,
            "I2_percent": np.nan,
            "Q": np.nan,
        }

    fixed_weights = 1.0 / np.square(standard_errors)

    fixed_effect = (
        np.sum(fixed_weights * effects)
        / np.sum(fixed_weights)
    )

    Q = np.sum(
        fixed_weights
        * np.square(effects - fixed_effect)
    )

    degrees_freedom = k - 1

    C = (
        np.sum(fixed_weights)
        - (
            np.sum(np.square(fixed_weights))
            / np.sum(fixed_weights)
        )
    )

    if C <= 0:
        tau2 = 0.0
    else:
        tau2 = max(
            0.0,
            (Q - degrees_freedom) / C
        )

    random_weights = 1.0 / (
        np.square(standard_errors) + tau2
    )

    pooled_effect = (
        np.sum(random_weights * effects)
        / np.sum(random_weights)
    )

    pooled_se = np.sqrt(
        1.0 / np.sum(random_weights)
    )

    pooled_z = pooled_effect / pooled_se
    pooled_p = 2.0 * norm.sf(abs(pooled_z))

    if Q <= 0:
        I2 = 0.0
    else:
        I2 = max(
            0.0,
            ((Q - degrees_freedom) / Q) * 100.0
        )

    return {
        "random_effect_log2FC": float(pooled_effect),
        "random_effect_SE": float(pooled_se),
        "random_effect_z": float(pooled_z),
        "random_effect_p": float(pooled_p),
        "tau2": float(tau2),
        "I2_percent": float(I2),
        "Q": float(Q),
    }


benchmark_rows = []

for gene, group in eligible.groupby(
    "gene_symbol",
    sort=True
):
    group = group.sort_values("cohort").copy()

    effects = group["log2FC"].to_numpy(dtype=float)
    standard_errors = group["SE"].to_numpy(dtype=float)
    p_values = group["p_value"].to_numpy(dtype=float)

    n_cohorts = len(group)

    n_up = int(np.sum(effects > 0))
    n_down = int(np.sum(effects < 0))
    n_zero = int(np.sum(effects == 0))

    if n_up > n_down:
        consensus_direction = "UP"
    elif n_down > n_up:
        consensus_direction = "DOWN"
    else:
        consensus_direction = "TIE"

    direction_vote_score = (
        max(n_up, n_down) / n_cohorts
        if n_cohorts >= 2
        else np.nan
    )

    fisher_statistic, fisher_p = fisher_combination(
        p_values
    )

    stouffer_z, stouffer_p = signed_stouffer_combination(
        p_values,
        effects
    )

    random_effects = dersimonian_laird_random_effects(
        effects,
        standard_errors
    )

    benchmark_rows.append(
        {
            "gene_symbol": gene,
            "benchmark_n_cohorts": n_cohorts,
            "benchmark_cohorts": ";".join(
                group["cohort"].astype(str).tolist()
            ),
            "n_up_cohorts": n_up,
            "n_down_cohorts": n_down,
            "n_zero_cohorts": n_zero,
            "consensus_direction": consensus_direction,
            "direction_vote_score": direction_vote_score,
            "fisher_statistic": fisher_statistic,
            "fisher_combined_p": fisher_p,
            "fisher_minus_log10_p": safe_minus_log10(
                fisher_p
            ),
            "stouffer_signed_z": stouffer_z,
            "stouffer_combined_p": stouffer_p,
            "stouffer_minus_log10_p": safe_minus_log10(
                stouffer_p
            ),
            **random_effects,
            "random_effect_minus_log10_p": safe_minus_log10(
                random_effects["random_effect_p"]
            ),
        }
    )

benchmark = pd.DataFrame(benchmark_rows)

# Merge with the locked original MCI components.
benchmark = primary_mci.merge(
    benchmark,
    on="gene_symbol",
    how="left",
    validate="one_to_one"
)

# Equal-weight alternative:
# each original MCI component receives weight 1/3.
benchmark["equal_weight_composite"] = (
    benchmark[
        [
            "D_g_direction_agreement",
            "S_g_effect_size_consistency",
            "R_g_statistical_reproducibility",
        ]
    ].mean(
        axis=1,
        skipna=False
    )
)

# Leave original insufficient-coverage genes without an equal-weight score.
benchmark.loc[
    benchmark["n_hcm_cohorts_available"] < 2,
    "equal_weight_composite"
] = np.nan


def descending_rank(series):
    """
    Rank 1 represents the strongest score.
    Ties receive average ranks.
    """
    return series.rank(
        ascending=False,
        method="average",
        na_option="bottom"
    )


ranking_columns = {
    "MCI": "rank_original_MCI",
    "equal_weight_composite": "rank_equal_weight",
    "direction_vote_score": "rank_direction_vote",
    "fisher_minus_log10_p": "rank_fisher",
    "stouffer_minus_log10_p": "rank_stouffer",
    "random_effect_minus_log10_p": "rank_random_effects",
}

for score_column, rank_column in ranking_columns.items():
    benchmark[rank_column] = descending_rank(
        benchmark[score_column]
    )

# Tier assignment for the equal-weight composite uses the same original
# thresholds only as a direct sensitivity comparison.
def assign_same_threshold_tier(score, n_cohorts):
    if n_cohorts < 2 or pd.isna(score):
        return "INSUFFICIENT_COVERAGE"
    if score >= 0.70:
        return "HIGH"
    if score >= 0.45:
        return "MODERATE"
    return "UNSTABLE"


benchmark["equal_weight_tier_same_thresholds"] = benchmark.apply(
    lambda row: assign_same_threshold_tier(
        row["equal_weight_composite"],
        row["n_hcm_cohorts_available"]
    ),
    axis=1
)

# Organize the most important columns near the front.
front_columns = [
    "gene_symbol",
    "disease_group",
    "stratum",
    "n_hcm_cohorts_available",
    "cohorts_available",
    "D_g_direction_agreement",
    "S_g_effect_size_consistency",
    "R_g_statistical_reproducibility",
    "MCI",
    "MCI_tier",
    "equal_weight_composite",
    "equal_weight_tier_same_thresholds",
    "direction_vote_score",
    "consensus_direction",
    "fisher_combined_p",
    "fisher_minus_log10_p",
    "stouffer_signed_z",
    "stouffer_combined_p",
    "stouffer_minus_log10_p",
    "random_effect_log2FC",
    "random_effect_SE",
    "random_effect_z",
    "random_effect_p",
    "random_effect_minus_log10_p",
    "tau2",
    "I2_percent",
    "rank_original_MCI",
    "rank_equal_weight",
    "rank_direction_vote",
    "rank_fisher",
    "rank_stouffer",
    "rank_random_effects",
]

remaining_columns = [
    column
    for column in benchmark.columns
    if column not in front_columns
]

benchmark = benchmark[
    front_columns + remaining_columns
]

output_path = (
    OUTPUT_DIR
    / "TASK11_STEP4_METHOD_BENCHMARK_GENE_LEVEL.csv"
)

benchmark.to_csv(
    output_path,
    index=False
)

print("=" * 105)
print("BENCHMARK DATASET CREATED")
print("=" * 105)
print(f"Output file: {output_path}")
print(f"Shape: {benchmark.shape}")
print(f"Unique genes: {benchmark['gene_symbol'].nunique()}")

print("\n" + "=" * 105)
print("METHOD COVERAGE")
print("=" * 105)

coverage_columns = [
    "MCI",
    "equal_weight_composite",
    "direction_vote_score",
    "fisher_minus_log10_p",
    "stouffer_minus_log10_p",
    "random_effect_minus_log10_p",
]

for column in coverage_columns:
    print(
        f"{column:>30}: "
        f"{benchmark[column].notna().sum()} genes"
    )

print("\n" + "=" * 105)
print("SCORE RANGES")
print("=" * 105)

for column in coverage_columns:
    valid = benchmark[column].dropna()

    if len(valid) > 0:
        print(
            f"{column:>30}: "
            f"min={valid.min():.6f}, "
            f"median={valid.median():.6f}, "
            f"max={valid.max():.6f}"
        )

print("\n" + "=" * 105)
print("TOP 10 GENES BY EACH METHOD")
print("=" * 105)

display_methods = {
    "Original MCI": "MCI",
    "Equal-weight composite": "equal_weight_composite",
    "Direction vote": "direction_vote_score",
    "Fisher combined significance": "fisher_minus_log10_p",
    "Signed Stouffer significance": "stouffer_minus_log10_p",
    "Random-effects meta-analysis": "random_effect_minus_log10_p",
}

for method_name, score_column in display_methods.items():
    top = (
        benchmark.loc[
            benchmark[score_column].notna(),
            ["gene_symbol", score_column]
        ]
        .sort_values(
            score_column,
            ascending=False
        )
        .head(10)
    )

    gene_list = ", ".join(
        top["gene_symbol"].tolist()
    )

    print(f"\n{method_name}:")
    print(gene_list)

print("\n" + "=" * 105)
print("EQUAL-WEIGHT TIER DISTRIBUTION")
print("=" * 105)

print(
    benchmark[
        "equal_weight_tier_same_thresholds"
    ]
    .value_counts(dropna=False)
    .to_string()
)

print("\nSTEP 4 COMPLETE")


In [ ]:
# STEP 5 — Quantify ranking agreement between MCI and benchmark methods
#
# Outputs:
# 1. Spearman rank correlation with original MCI
# 2. Kendall rank correlation with original MCI
# 3. Mean, median, and maximum rank displacement
# 4. Top-10 and top-18 overlap, including ties
# 5. Equal-weight tier agreement and Cohen's kappa
# 6. Gene-level rank displacement table

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr

PROJECT_DIR = PROJECT_DIR
OUTPUT_DIR = PROJECT_DIR / "results" / "task11_benchmarking"

benchmark_path = (
    OUTPUT_DIR
    / "TASK11_STEP4_METHOD_BENCHMARK_GENE_LEVEL.csv"
)

benchmark = pd.read_csv(benchmark_path)

method_columns = {
    "Original MCI": "MCI",
    "Equal-weight composite": "equal_weight_composite",
    "Direction vote": "direction_vote_score",
    "Fisher combined significance": "fisher_minus_log10_p",
    "Signed Stouffer significance": "stouffer_minus_log10_p",
    "Random-effects meta-analysis": "random_effect_minus_log10_p",
}

rank_columns = {
    "Original MCI": "rank_original_MCI",
    "Equal-weight composite": "rank_equal_weight",
    "Direction vote": "rank_direction_vote",
    "Fisher combined significance": "rank_fisher",
    "Signed Stouffer significance": "rank_stouffer",
    "Random-effects meta-analysis": "rank_random_effects",
}

# Restrict ranking comparisons to genes with a valid original MCI.
analysis = benchmark.loc[
    benchmark["MCI"].notna()
].copy()

print("=" * 110)
print("ANALYSIS SET")
print("=" * 110)
print(f"Genes with valid original MCI: {len(analysis)}")
print(
    "Original tier distribution:\n"
    + analysis["MCI_tier"].value_counts().to_string()
)


def inclusive_top_set(data, score_column, k):
    """
    Return all genes at or above the kth-highest score.

    This includes ties at the cutoff rather than arbitrarily breaking them.
    """
    valid = data.loc[
        data[score_column].notna(),
        ["gene_symbol", score_column]
    ].copy()

    valid = valid.sort_values(
        [score_column, "gene_symbol"],
        ascending=[False, True]
    )

    if len(valid) == 0:
        return set(), np.nan

    effective_k = min(k, len(valid))
    cutoff = valid.iloc[effective_k - 1][score_column]

    selected = set(
        valid.loc[
            valid[score_column] >= cutoff,
            "gene_symbol"
        ]
    )

    return selected, float(cutoff)


# Original reference sets
original_top10, original_top10_cutoff = inclusive_top_set(
    analysis,
    "MCI",
    10
)

original_high_set = set(
    analysis.loc[
        analysis["MCI_tier"].astype(str).str.upper() == "HIGH",
        "gene_symbol"
    ]
)

n_original_high = len(original_high_set)

print("\n" + "=" * 110)
print("REFERENCE SETS")
print("=" * 110)
print(f"Original inclusive top-10 size: {len(original_top10)}")
print(f"Original top-10 MCI cutoff: {original_top10_cutoff:.6f}")
print(f"Original HIGH-tier size: {n_original_high}")


summary_rows = []
gene_rank_rows = []

for method_name, score_column in method_columns.items():

    if method_name == "Original MCI":
        continue

    rank_column = rank_columns[method_name]

    valid = analysis.loc[
        analysis["MCI"].notna()
        & analysis[score_column].notna()
        & analysis["rank_original_MCI"].notna()
        & analysis[rank_column].notna()
    ].copy()

    # Correlations are calculated on score values.
    spearman_result = spearmanr(
        valid["MCI"],
        valid[score_column],
        nan_policy="omit"
    )

    kendall_result = kendalltau(
        valid["MCI"],
        valid[score_column],
        nan_policy="omit"
    )

    valid["rank_displacement"] = (
        valid[rank_column]
        - valid["rank_original_MCI"]
    )

    valid["absolute_rank_displacement"] = (
        valid["rank_displacement"].abs()
    )

    # Inclusive top-10 comparison
    method_top10, method_top10_cutoff = inclusive_top_set(
        valid,
        score_column,
        10
    )

    top10_overlap = len(
        original_top10.intersection(method_top10)
    )

    top10_union = len(
        original_top10.union(method_top10)
    )

    top10_jaccard = (
        top10_overlap / top10_union
        if top10_union > 0
        else np.nan
    )

    top10_original_recall = (
        top10_overlap / len(original_top10)
        if len(original_top10) > 0
        else np.nan
    )

    # Compare each benchmark's top-N set to the original HIGH tier,
    # where N equals the original number of HIGH genes.
    method_top_high_n, method_top_high_cutoff = inclusive_top_set(
        valid,
        score_column,
        n_original_high
    )

    high_overlap = len(
        original_high_set.intersection(method_top_high_n)
    )

    high_union = len(
        original_high_set.union(method_top_high_n)
    )

    high_jaccard = (
        high_overlap / high_union
        if high_union > 0
        else np.nan
    )

    high_recall = (
        high_overlap / len(original_high_set)
        if len(original_high_set) > 0
        else np.nan
    )

    # Count unique score values to evaluate method discrimination.
    unique_score_count = valid[score_column].nunique(dropna=True)
    largest_tie_size = int(
        valid.groupby(score_column)["gene_symbol"]
        .size()
        .max()
    )

    summary_rows.append(
        {
            "method": method_name,
            "n_genes_compared": len(valid),
            "unique_score_values": unique_score_count,
            "largest_tie_size": largest_tie_size,
            "spearman_rho_vs_MCI": spearman_result.statistic,
            "spearman_p_value": spearman_result.pvalue,
            "kendall_tau_vs_MCI": kendall_result.statistic,
            "kendall_p_value": kendall_result.pvalue,
            "mean_absolute_rank_shift": (
                valid["absolute_rank_displacement"].mean()
            ),
            "median_absolute_rank_shift": (
                valid["absolute_rank_displacement"].median()
            ),
            "maximum_absolute_rank_shift": (
                valid["absolute_rank_displacement"].max()
            ),
            "method_top10_cutoff": method_top10_cutoff,
            "method_inclusive_top10_size": len(method_top10),
            "top10_overlap_with_MCI": top10_overlap,
            "top10_original_recall": top10_original_recall,
            "top10_jaccard": top10_jaccard,
            "method_top_high_cutoff": method_top_high_cutoff,
            "method_top_high_inclusive_size": len(method_top_high_n),
            "original_HIGH_overlap": high_overlap,
            "original_HIGH_recall": high_recall,
            "original_HIGH_jaccard": high_jaccard,
        }
    )

    for _, row in valid.iterrows():
        gene_rank_rows.append(
            {
                "gene_symbol": row["gene_symbol"],
                "method": method_name,
                "original_MCI": row["MCI"],
                "benchmark_score": row[score_column],
                "original_MCI_rank": row["rank_original_MCI"],
                "benchmark_rank": row[rank_column],
                "rank_displacement": row["rank_displacement"],
                "absolute_rank_displacement": (
                    row["absolute_rank_displacement"]
                ),
            }
        )

summary = pd.DataFrame(summary_rows)
gene_rank_displacement = pd.DataFrame(gene_rank_rows)

# Sort from highest to lowest rank agreement.
summary = summary.sort_values(
    "spearman_rho_vs_MCI",
    ascending=False
).reset_index(drop=True)

# -------------------------------------------------------------------
# Equal-weight tier agreement
# -------------------------------------------------------------------

def normalize_tier(value):
    if pd.isna(value):
        return "MISSING"

    normalized = (
        str(value)
        .strip()
        .upper()
        .replace("-", "_")
        .replace(" ", "_")
    )

    if "INSUFFICIENT" in normalized:
        return "INSUFFICIENT_COVERAGE"

    return normalized


tier_analysis = analysis.loc[
    analysis["equal_weight_composite"].notna()
].copy()

tier_analysis["original_tier_normalized"] = (
    tier_analysis["MCI_tier"].apply(normalize_tier)
)

tier_analysis["equal_weight_tier_normalized"] = (
    tier_analysis[
        "equal_weight_tier_same_thresholds"
    ].apply(normalize_tier)
)

tier_categories = [
    "HIGH",
    "MODERATE",
    "UNSTABLE",
]

tier_confusion = pd.crosstab(
    tier_analysis["original_tier_normalized"],
    tier_analysis["equal_weight_tier_normalized"],
    rownames=["Original MCI tier"],
    colnames=["Equal-weight tier"],
    dropna=False
).reindex(
    index=tier_categories,
    columns=tier_categories,
    fill_value=0
)

exact_tier_matches = int(
    (
        tier_analysis["original_tier_normalized"]
        == tier_analysis["equal_weight_tier_normalized"]
    ).sum()
)

tier_match_rate = (
    exact_tier_matches / len(tier_analysis)
)

# Manual unweighted Cohen's kappa
observed_agreement = tier_match_rate

original_proportions = (
    tier_analysis["original_tier_normalized"]
    .value_counts(normalize=True)
)

equal_proportions = (
    tier_analysis["equal_weight_tier_normalized"]
    .value_counts(normalize=True)
)

expected_agreement = sum(
    original_proportions.get(category, 0.0)
    * equal_proportions.get(category, 0.0)
    for category in tier_categories
)

if np.isclose(1.0 - expected_agreement, 0.0):
    cohens_kappa = np.nan
else:
    cohens_kappa = (
        observed_agreement - expected_agreement
    ) / (
        1.0 - expected_agreement
    )

tier_changes = tier_analysis.loc[
    tier_analysis["original_tier_normalized"]
    != tier_analysis["equal_weight_tier_normalized"],
    [
        "gene_symbol",
        "MCI",
        "original_tier_normalized",
        "equal_weight_composite",
        "equal_weight_tier_normalized",
    ]
].sort_values(
    ["original_tier_normalized", "gene_symbol"]
)

# -------------------------------------------------------------------
# Save outputs
# -------------------------------------------------------------------

summary_path = (
    OUTPUT_DIR
    / "TASK11_STEP5_RANKING_AGREEMENT_SUMMARY.csv"
)

gene_rank_path = (
    OUTPUT_DIR
    / "TASK11_STEP5_GENE_LEVEL_RANK_DISPLACEMENT.csv"
)

tier_confusion_path = (
    OUTPUT_DIR
    / "TASK11_STEP5_EQUAL_WEIGHT_TIER_CONFUSION.csv"
)

tier_changes_path = (
    OUTPUT_DIR
    / "TASK11_STEP5_EQUAL_WEIGHT_TIER_CHANGES.csv"
)

summary.to_csv(summary_path, index=False)
gene_rank_displacement.to_csv(gene_rank_path, index=False)
tier_confusion.to_csv(tier_confusion_path)
tier_changes.to_csv(tier_changes_path, index=False)

print("\n" + "=" * 110)
print("RANKING AGREEMENT WITH ORIGINAL MCI")
print("=" * 110)

display_columns = [
    "method",
    "n_genes_compared",
    "unique_score_values",
    "largest_tie_size",
    "spearman_rho_vs_MCI",
    "kendall_tau_vs_MCI",
    "mean_absolute_rank_shift",
    "median_absolute_rank_shift",
    "maximum_absolute_rank_shift",
]

print(
    summary[display_columns]
    .round(4)
    .to_string(index=False)
)

print("\n" + "=" * 110)
print("TOP-10 OVERLAP, INCLUDING TIES AT THE CUTOFF")
print("=" * 110)

top10_columns = [
    "method",
    "method_inclusive_top10_size",
    "top10_overlap_with_MCI",
    "top10_original_recall",
    "top10_jaccard",
]

print(
    summary[top10_columns]
    .round(4)
    .to_string(index=False)
)

print("\n" + "=" * 110)
print(f"OVERLAP WITH ORIGINAL HIGH TIER — {n_original_high} GENES")
print("=" * 110)

high_columns = [
    "method",
    "method_top_high_inclusive_size",
    "original_HIGH_overlap",
    "original_HIGH_recall",
    "original_HIGH_jaccard",
]

print(
    summary[high_columns]
    .round(4)
    .to_string(index=False)
)

print("\n" + "=" * 110)
print("EQUAL-WEIGHT TIER AGREEMENT")
print("=" * 110)

print(f"Genes compared: {len(tier_analysis)}")
print(f"Exact tier matches: {exact_tier_matches}")
print(f"Exact tier agreement: {tier_match_rate:.4f}")
print(f"Cohen's kappa: {cohens_kappa:.4f}")

print("\nTier confusion matrix:")
print(tier_confusion.to_string())

print("\nGenes changing tier under equal weights:")
if len(tier_changes) == 0:
    print("None")
else:
    print(tier_changes.to_string(index=False))

print("\n" + "=" * 110)
print("OUTPUT FILES")
print("=" * 110)
print(summary_path)
print(gene_rank_path)
print(tier_confusion_path)
print(tier_changes_path)

print("\nSTEP 5 COMPLETE")


In [ ]:
# STEP 6 — Adversarial weight sensitivity and leave-one-component-out analysis
#
# This step tests:
# - equal weights,
# - moderate alternative weight combinations,
# - strongly component-heavy combinations,
# - removal of D_g,
# - removal of S_g,
# - removal of R_g.
#
# The original thresholds are held fixed at:
# MODERATE >= 0.45
# HIGH >= 0.70
#
# Holding thresholds fixed isolates the effect of changing weights.
# Threshold sensitivity will be tested separately in the next step.

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr

PROJECT_DIR = PROJECT_DIR
OUTPUT_DIR = PROJECT_DIR / "results" / "task11_benchmarking"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

benchmark_path = (
    OUTPUT_DIR
    / "TASK11_STEP4_METHOD_BENCHMARK_GENE_LEVEL.csv"
)

benchmark = pd.read_csv(benchmark_path)

D_COL = "D_g_direction_agreement"
S_COL = "S_g_effect_size_consistency"
R_COL = "R_g_statistical_reproducibility"

# Only genes with enough coverage for the original MCI are used.
analysis = benchmark.loc[
    benchmark["MCI"].notna()
    & benchmark[D_COL].notna()
    & benchmark[S_COL].notna()
    & benchmark[R_COL].notna()
].copy()

# ---------------------------------------------------------------------
# Define weight scenarios
# ---------------------------------------------------------------------
#
# "Heavy" scenarios deliberately assign 55% to one component.
#
# Leave-one-component-out scenarios remove one component and renormalize
# the two remaining original weights so that the total remains 1.0.

weight_scenarios = {
    "PRIMARY_40D_35S_25R": {
        "D_weight": 0.40,
        "S_weight": 0.35,
        "R_weight": 0.25,
        "scenario_type": "Primary",
    },

    "EQUAL_33D_33S_33R": {
        "D_weight": 1 / 3,
        "S_weight": 1 / 3,
        "R_weight": 1 / 3,
        "scenario_type": "Equal weight",
    },

    "ALT_45D_35S_20R": {
        "D_weight": 0.45,
        "S_weight": 0.35,
        "R_weight": 0.20,
        "scenario_type": "Plausible alternative",
    },

    "ALT_35D_40S_25R": {
        "D_weight": 0.35,
        "S_weight": 0.40,
        "R_weight": 0.25,
        "scenario_type": "Plausible alternative",
    },

    "ALT_35D_30S_35R": {
        "D_weight": 0.35,
        "S_weight": 0.30,
        "R_weight": 0.35,
        "scenario_type": "Plausible alternative",
    },

    "DIRECTION_HEAVY_55D_25S_20R": {
        "D_weight": 0.55,
        "S_weight": 0.25,
        "R_weight": 0.20,
        "scenario_type": "Component-heavy adversarial",
    },

    "CONSISTENCY_HEAVY_25D_55S_20R": {
        "D_weight": 0.25,
        "S_weight": 0.55,
        "R_weight": 0.20,
        "scenario_type": "Component-heavy adversarial",
    },

    "REPRODUCIBILITY_HEAVY_25D_20S_55R": {
        "D_weight": 0.25,
        "S_weight": 0.20,
        "R_weight": 0.55,
        "scenario_type": "Component-heavy adversarial",
    },

    # Remove R and retain original D:S relative weighting.
    "LEAVE_OUT_R_RENORMALIZED": {
        "D_weight": 0.40 / (0.40 + 0.35),
        "S_weight": 0.35 / (0.40 + 0.35),
        "R_weight": 0.00,
        "scenario_type": "Leave-one-component-out",
    },

    # Remove S and retain original D:R relative weighting.
    "LEAVE_OUT_S_RENORMALIZED": {
        "D_weight": 0.40 / (0.40 + 0.25),
        "S_weight": 0.00,
        "R_weight": 0.25 / (0.40 + 0.25),
        "scenario_type": "Leave-one-component-out",
    },

    # Remove D and retain original S:R relative weighting.
    "LEAVE_OUT_D_RENORMALIZED": {
        "D_weight": 0.00,
        "S_weight": 0.35 / (0.35 + 0.25),
        "R_weight": 0.25 / (0.35 + 0.25),
        "scenario_type": "Leave-one-component-out",
    },
}


def assign_tier(score):
    if pd.isna(score):
        return "INSUFFICIENT_COVERAGE"

    if score >= 0.70:
        return "HIGH"

    if score >= 0.45:
        return "MODERATE"

    return "UNSTABLE"


def descending_rank(series):
    return series.rank(
        ascending=False,
        method="average",
        na_option="bottom"
    )


def inclusive_top_set(data, score_column, k):
    """
    Select all genes with scores at or above the kth-highest value,
    including ties at the cutoff.
    """
    valid = data.loc[
        data[score_column].notna(),
        ["gene_symbol", score_column]
    ].copy()

    valid = valid.sort_values(
        [score_column, "gene_symbol"],
        ascending=[False, True]
    )

    effective_k = min(k, len(valid))
    cutoff = valid.iloc[effective_k - 1][score_column]

    selected = set(
        valid.loc[
            valid[score_column] >= cutoff,
            "gene_symbol"
        ]
    )

    return selected, float(cutoff)


# Validate scenario weights before running the analysis.
print("=" * 115)
print("WEIGHT SCENARIO VALIDATION")
print("=" * 115)

for scenario_name, values in weight_scenarios.items():
    weight_sum = (
        values["D_weight"]
        + values["S_weight"]
        + values["R_weight"]
    )

    if not np.isclose(weight_sum, 1.0):
        raise ValueError(
            f"Weights do not sum to 1 for {scenario_name}: "
            f"{weight_sum}"
        )

    print(
        f"{scenario_name:42} | "
        f"D={values['D_weight']:.4f}, "
        f"S={values['S_weight']:.4f}, "
        f"R={values['R_weight']:.4f}, "
        f"sum={weight_sum:.4f}"
    )

# ---------------------------------------------------------------------
# Calculate scenario scores, tiers, and ranks
# ---------------------------------------------------------------------

gene_level = analysis[
    [
        "gene_symbol",
        D_COL,
        S_COL,
        R_COL,
        "MCI",
        "MCI_tier",
    ]
].copy()

scenario_score_columns = []
scenario_tier_columns = []
scenario_rank_columns = []

for scenario_name, weights in weight_scenarios.items():

    score_column = f"{scenario_name}__score"
    tier_column = f"{scenario_name}__tier"
    rank_column = f"{scenario_name}__rank"

    gene_level[score_column] = (
        weights["D_weight"] * gene_level[D_COL]
        + weights["S_weight"] * gene_level[S_COL]
        + weights["R_weight"] * gene_level[R_COL]
    )

    gene_level[tier_column] = (
        gene_level[score_column].apply(assign_tier)
    )

    gene_level[rank_column] = descending_rank(
        gene_level[score_column]
    )

    scenario_score_columns.append(score_column)
    scenario_tier_columns.append(tier_column)
    scenario_rank_columns.append(rank_column)

primary_score_col = "PRIMARY_40D_35S_25R__score"
primary_tier_col = "PRIMARY_40D_35S_25R__tier"
primary_rank_col = "PRIMARY_40D_35S_25R__rank"

# Verify that the primary scenario reproduces the existing MCI.
primary_max_difference = (
    gene_level[primary_score_col]
    - gene_level["MCI"]
).abs().max()

primary_tier_match = (
    gene_level[primary_tier_col]
    == gene_level["MCI_tier"]
).mean()

if primary_max_difference > 1e-10:
    raise ValueError(
        "Primary weight scenario did not reproduce the original MCI."
    )

# ---------------------------------------------------------------------
# Scenario-level summary
# ---------------------------------------------------------------------

original_high_genes = set(
    gene_level.loc[
        gene_level[primary_tier_col] == "HIGH",
        "gene_symbol"
    ]
)

n_original_high = len(original_high_genes)

summary_rows = []
transition_rows = []

for scenario_name, weights in weight_scenarios.items():

    score_column = f"{scenario_name}__score"
    tier_column = f"{scenario_name}__tier"
    rank_column = f"{scenario_name}__rank"

    spearman_result = spearmanr(
        gene_level[primary_score_col],
        gene_level[score_column]
    )

    kendall_result = kendalltau(
        gene_level[primary_score_col],
        gene_level[score_column]
    )

    absolute_rank_shift = (
        gene_level[rank_column]
        - gene_level[primary_rank_col]
    ).abs()

    exact_tier_matches = (
        gene_level[tier_column]
        == gene_level[primary_tier_col]
    )

    n_tier_changes = int((~exact_tier_matches).sum())
    pct_tier_changes = (
        100.0 * n_tier_changes / len(gene_level)
    )

    scenario_top_n, top_n_cutoff = inclusive_top_set(
        gene_level,
        score_column,
        n_original_high
    )

    high_overlap = len(
        original_high_genes.intersection(scenario_top_n)
    )

    high_union = len(
        original_high_genes.union(scenario_top_n)
    )

    summary_rows.append(
        {
            "scenario": scenario_name,
            "scenario_type": weights["scenario_type"],
            "D_weight": weights["D_weight"],
            "S_weight": weights["S_weight"],
            "R_weight": weights["R_weight"],
            "n_genes": len(gene_level),
            "score_min": gene_level[score_column].min(),
            "score_median": gene_level[score_column].median(),
            "score_max": gene_level[score_column].max(),
            "unique_score_values": (
                gene_level[score_column].nunique()
            ),
            "spearman_rho_vs_primary": (
                spearman_result.statistic
            ),
            "spearman_p_value": spearman_result.pvalue,
            "kendall_tau_vs_primary": (
                kendall_result.statistic
            ),
            "kendall_p_value": kendall_result.pvalue,
            "mean_absolute_rank_shift": (
                absolute_rank_shift.mean()
            ),
            "median_absolute_rank_shift": (
                absolute_rank_shift.median()
            ),
            "maximum_absolute_rank_shift": (
                absolute_rank_shift.max()
            ),
            "exact_tier_matches": int(
                exact_tier_matches.sum()
            ),
            "n_tier_changes": n_tier_changes,
            "percent_tier_changes": pct_tier_changes,
            "n_HIGH": int(
                (gene_level[tier_column] == "HIGH").sum()
            ),
            "n_MODERATE": int(
                (gene_level[tier_column] == "MODERATE").sum()
            ),
            "n_UNSTABLE": int(
                (gene_level[tier_column] == "UNSTABLE").sum()
            ),
            "top_n_cutoff": top_n_cutoff,
            "top_n_inclusive_size": len(scenario_top_n),
            "original_HIGH_overlap": high_overlap,
            "original_HIGH_recall": (
                high_overlap / n_original_high
            ),
            "original_HIGH_jaccard": (
                high_overlap / high_union
                if high_union > 0
                else np.nan
            ),
        }
    )

    transition_table = pd.crosstab(
        gene_level[primary_tier_col],
        gene_level[tier_column],
        rownames=["primary_tier"],
        colnames=["scenario_tier"]
    )

    for primary_tier in [
        "HIGH",
        "MODERATE",
        "UNSTABLE",
    ]:
        for scenario_tier in [
            "HIGH",
            "MODERATE",
            "UNSTABLE",
        ]:
            transition_rows.append(
                {
                    "scenario": scenario_name,
                    "scenario_type": weights[
                        "scenario_type"
                    ],
                    "primary_tier": primary_tier,
                    "scenario_tier": scenario_tier,
                    "n_genes": int(
                        transition_table
                        .reindex(
                            index=[
                                "HIGH",
                                "MODERATE",
                                "UNSTABLE",
                            ],
                            columns=[
                                "HIGH",
                                "MODERATE",
                                "UNSTABLE",
                            ],
                            fill_value=0
                        )
                        .loc[
                            primary_tier,
                            scenario_tier
                        ]
                    ),
                }
            )

summary = pd.DataFrame(summary_rows)

summary["scenario_order"] = summary["scenario"].map(
    {
        name: index
        for index, name in enumerate(
            weight_scenarios.keys()
        )
    }
)

summary = (
    summary
    .sort_values("scenario_order")
    .drop(columns="scenario_order")
    .reset_index(drop=True)
)

transitions = pd.DataFrame(transition_rows)

# ---------------------------------------------------------------------
# Gene-level robustness envelope across non-primary scenarios
# ---------------------------------------------------------------------

non_primary_score_columns = [
    column
    for column in scenario_score_columns
    if column != primary_score_col
]

non_primary_tier_columns = [
    column
    for column in scenario_tier_columns
    if column != primary_tier_col
]

gene_level["minimum_sensitivity_score"] = (
    gene_level[non_primary_score_columns].min(axis=1)
)

gene_level["maximum_sensitivity_score"] = (
    gene_level[non_primary_score_columns].max(axis=1)
)

gene_level["sensitivity_score_range"] = (
    gene_level["maximum_sensitivity_score"]
    - gene_level["minimum_sensitivity_score"]
)

gene_level["n_distinct_tiers_across_all_scenarios"] = (
    gene_level[scenario_tier_columns]
    .nunique(axis=1)
)

gene_level["n_nonprimary_scenarios_changing_tier"] = (
    gene_level[non_primary_tier_columns]
    .ne(
        gene_level[primary_tier_col],
        axis=0
    )
    .sum(axis=1)
)

gene_level["any_weight_sensitivity_tier_change"] = (
    gene_level[
        "n_nonprimary_scenarios_changing_tier"
    ] > 0
)

gene_level["tier_stable_across_all_weight_scenarios"] = (
    gene_level[
        "n_distinct_tiers_across_all_scenarios"
    ] == 1
)

gene_level = gene_level.sort_values(
    [
        "n_nonprimary_scenarios_changing_tier",
        "sensitivity_score_range",
        "MCI",
    ],
    ascending=[False, False, False]
)

# ---------------------------------------------------------------------
# Save outputs
# ---------------------------------------------------------------------

summary_path = (
    OUTPUT_DIR
    / "TASK11_STEP6_WEIGHT_SENSITIVITY_SUMMARY.csv"
)

gene_level_path = (
    OUTPUT_DIR
    / "TASK11_STEP6_WEIGHT_SENSITIVITY_GENE_LEVEL.csv"
)

transition_path = (
    OUTPUT_DIR
    / "TASK11_STEP6_WEIGHT_SENSITIVITY_TIER_TRANSITIONS.csv"
)

summary.to_csv(summary_path, index=False)
gene_level.to_csv(gene_level_path, index=False)
transitions.to_csv(transition_path, index=False)

# ---------------------------------------------------------------------
# Print review output
# ---------------------------------------------------------------------

print("\n" + "=" * 115)
print("PRIMARY REPRODUCTION CHECK")
print("=" * 115)
print(
    "Maximum difference between reconstructed primary scenario "
    f"and original MCI: {primary_max_difference:.12g}"
)
print(
    "Primary scenario tier agreement with original MCI: "
    f"{primary_tier_match:.4f}"
)

print("\n" + "=" * 115)
print("WEIGHT SENSITIVITY SUMMARY")
print("=" * 115)

summary_display_columns = [
    "scenario",
    "scenario_type",
    "spearman_rho_vs_primary",
    "kendall_tau_vs_primary",
    "mean_absolute_rank_shift",
    "maximum_absolute_rank_shift",
    "n_tier_changes",
    "percent_tier_changes",
    "n_HIGH",
    "n_MODERATE",
    "n_UNSTABLE",
    "original_HIGH_overlap",
    "original_HIGH_recall",
]

print(
    summary[summary_display_columns]
    .round(4)
    .to_string(index=False)
)

print("\n" + "=" * 115)
print("OVERALL GENE-LEVEL WEIGHT ROBUSTNESS")
print("=" * 115)

n_fully_stable = int(
    gene_level[
        "tier_stable_across_all_weight_scenarios"
    ].sum()
)

n_any_change = int(
    gene_level[
        "any_weight_sensitivity_tier_change"
    ].sum()
)

print(f"Genes assessed: {len(gene_level)}")
print(
    "Genes retaining the same tier under every weight scenario: "
    f"{n_fully_stable}/{len(gene_level)} "
    f"({100 * n_fully_stable / len(gene_level):.1f}%)"
)
print(
    "Genes changing tier under at least one alternative scenario: "
    f"{n_any_change}/{len(gene_level)} "
    f"({100 * n_any_change / len(gene_level):.1f}%)"
)

print("\n" + "=" * 115)
print("GENES MOST SENSITIVE TO WEIGHT CHANGES")
print("=" * 115)

sensitive_columns = [
    "gene_symbol",
    "MCI",
    "MCI_tier",
    "minimum_sensitivity_score",
    "maximum_sensitivity_score",
    "sensitivity_score_range",
    "n_distinct_tiers_across_all_scenarios",
    "n_nonprimary_scenarios_changing_tier",
]

sensitive_genes = gene_level.loc[
    gene_level[
        "any_weight_sensitivity_tier_change"
    ],
    sensitive_columns
].copy()

if len(sensitive_genes) == 0:
    print("No genes changed tier under any tested weight scenario.")
else:
    print(
        sensitive_genes
        .round(4)
        .to_string(index=False)
    )

print("\n" + "=" * 115)
print("OUTPUT FILES")
print("=" * 115)
print(summary_path)
print(gene_level_path)
print(transition_path)

print("\nSTEP 6 COMPLETE")


In [ ]:
# STEP 7 — Adversarial threshold sensitivity analysis
#
# Tests the original MCI under combinations of:
# MODERATE threshold: 0.40, 0.425, 0.45, 0.475, 0.50
# HIGH threshold:     0.65, 0.675, 0.70, 0.725, 0.75
#
# The MCI scores remain unchanged. Only classification thresholds change.
#
# Outputs:
# 1. Scenario-level tier changes
# 2. Gene-level threshold robustness
# 3. Tier-transition counts
# 4. Full gene-by-threshold tier matrix

from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd

PROJECT_DIR = PROJECT_DIR
OUTPUT_DIR = PROJECT_DIR / "results" / "task11_benchmarking"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

benchmark_path = (
    OUTPUT_DIR
    / "TASK11_STEP4_METHOD_BENCHMARK_GENE_LEVEL.csv"
)

benchmark = pd.read_csv(benchmark_path)

# Restrict analysis to the 47 genes with valid primary MCI scores.
analysis = benchmark.loc[
    benchmark["MCI"].notna()
].copy()

PRIMARY_MODERATE_THRESHOLD = 0.45
PRIMARY_HIGH_THRESHOLD = 0.70

moderate_thresholds = [
    0.400,
    0.425,
    0.450,
    0.475,
    0.500,
]

high_thresholds = [
    0.650,
    0.675,
    0.700,
    0.725,
    0.750,
]


def assign_threshold_tier(
    score,
    moderate_threshold,
    high_threshold
):
    """
    Assign an MCI tier using scenario-specific thresholds.
    """
    if pd.isna(score):
        return "INSUFFICIENT_COVERAGE"

    if score >= high_threshold:
        return "HIGH"

    if score >= moderate_threshold:
        return "MODERATE"

    return "UNSTABLE"


def normalize_tier(value):
    if pd.isna(value):
        return "MISSING"

    value = (
        str(value)
        .strip()
        .upper()
        .replace("-", "_")
        .replace(" ", "_")
    )

    if "INSUFFICIENT" in value:
        return "INSUFFICIENT_COVERAGE"

    return value


analysis["primary_tier"] = (
    analysis["MCI_tier"].apply(normalize_tier)
)

# Verify that the locked thresholds reproduce the original tiers.
analysis["recreated_primary_tier"] = analysis["MCI"].apply(
    lambda score: assign_threshold_tier(
        score,
        PRIMARY_MODERATE_THRESHOLD,
        PRIMARY_HIGH_THRESHOLD
    )
)

primary_mismatches = int(
    (
        analysis["primary_tier"]
        != analysis["recreated_primary_tier"]
    ).sum()
)

if primary_mismatches != 0:
    raise ValueError(
        f"Primary threshold reconstruction produced "
        f"{primary_mismatches} tier mismatches."
    )

print("=" * 118)
print("PRIMARY THRESHOLD REPRODUCTION")
print("=" * 118)
print(f"Genes assessed: {len(analysis)}")
print(f"Primary tier mismatches: {primary_mismatches}")
print(
    "Primary thresholds: "
    f"MODERATE >= {PRIMARY_MODERATE_THRESHOLD:.3f}, "
    f"HIGH >= {PRIMARY_HIGH_THRESHOLD:.3f}"
)

# ---------------------------------------------------------------------
# Generate all valid threshold scenarios
# ---------------------------------------------------------------------

threshold_scenarios = []

for moderate_threshold, high_threshold in product(
    moderate_thresholds,
    high_thresholds
):
    if moderate_threshold >= high_threshold:
        continue

    scenario_name = (
        f"MOD_{moderate_threshold:.3f}"
        f"__HIGH_{high_threshold:.3f}"
    )

    threshold_scenarios.append(
        {
            "scenario": scenario_name,
            "moderate_threshold": moderate_threshold,
            "high_threshold": high_threshold,
            "is_primary_threshold_pair": (
                np.isclose(
                    moderate_threshold,
                    PRIMARY_MODERATE_THRESHOLD
                )
                and np.isclose(
                    high_threshold,
                    PRIMARY_HIGH_THRESHOLD
                )
            ),
        }
    )

print("\n" + "=" * 118)
print("THRESHOLD GRID")
print("=" * 118)
print(
    f"Moderate thresholds tested: {moderate_thresholds}"
)
print(
    f"High thresholds tested: {high_thresholds}"
)
print(
    f"Valid threshold combinations: "
    f"{len(threshold_scenarios)}"
)

# ---------------------------------------------------------------------
# Calculate tiers under each threshold scenario
# ---------------------------------------------------------------------

tier_matrix = analysis[
    [
        "gene_symbol",
        "MCI",
        "primary_tier",
    ]
].copy()

summary_rows = []
transition_rows = []

tier_order = [
    "HIGH",
    "MODERATE",
    "UNSTABLE",
]

for scenario in threshold_scenarios:

    scenario_name = scenario["scenario"]
    moderate_threshold = scenario[
        "moderate_threshold"
    ]
    high_threshold = scenario[
        "high_threshold"
    ]

    tier_column = f"{scenario_name}__tier"

    tier_matrix[tier_column] = tier_matrix["MCI"].apply(
        lambda score: assign_threshold_tier(
            score,
            moderate_threshold,
            high_threshold
        )
    )

    exact_matches = (
        tier_matrix[tier_column]
        == tier_matrix["primary_tier"]
    )

    changed = ~exact_matches

    n_changes = int(changed.sum())
    percent_changes = (
        100.0 * n_changes / len(tier_matrix)
    )

    # Direction of changes relative to original classification.
    primary_numeric = tier_matrix[
        "primary_tier"
    ].map(
        {
            "UNSTABLE": 0,
            "MODERATE": 1,
            "HIGH": 2,
        }
    )

    scenario_numeric = tier_matrix[
        tier_column
    ].map(
        {
            "UNSTABLE": 0,
            "MODERATE": 1,
            "HIGH": 2,
        }
    )

    n_promoted = int(
        (scenario_numeric > primary_numeric).sum()
    )

    n_demoted = int(
        (scenario_numeric < primary_numeric).sum()
    )

    confusion = pd.crosstab(
        tier_matrix["primary_tier"],
        tier_matrix[tier_column],
        rownames=["primary_tier"],
        colnames=["scenario_tier"]
    ).reindex(
        index=tier_order,
        columns=tier_order,
        fill_value=0
    )

    summary_rows.append(
        {
            "scenario": scenario_name,
            "moderate_threshold": moderate_threshold,
            "high_threshold": high_threshold,
            "is_primary_threshold_pair": scenario[
                "is_primary_threshold_pair"
            ],
            "n_genes": len(tier_matrix),
            "exact_tier_matches": int(
                exact_matches.sum()
            ),
            "n_tier_changes": n_changes,
            "percent_tier_changes": percent_changes,
            "n_promoted_relative_to_primary": n_promoted,
            "n_demoted_relative_to_primary": n_demoted,
            "n_HIGH": int(
                (tier_matrix[tier_column] == "HIGH").sum()
            ),
            "n_MODERATE": int(
                (
                    tier_matrix[tier_column]
                    == "MODERATE"
                ).sum()
            ),
            "n_UNSTABLE": int(
                (
                    tier_matrix[tier_column]
                    == "UNSTABLE"
                ).sum()
            ),
        }
    )

    for primary_tier in tier_order:
        for scenario_tier in tier_order:
            transition_rows.append(
                {
                    "scenario": scenario_name,
                    "moderate_threshold": (
                        moderate_threshold
                    ),
                    "high_threshold": high_threshold,
                    "primary_tier": primary_tier,
                    "scenario_tier": scenario_tier,
                    "n_genes": int(
                        confusion.loc[
                            primary_tier,
                            scenario_tier
                        ]
                    ),
                }
            )

summary = pd.DataFrame(summary_rows)
transitions = pd.DataFrame(transition_rows)

summary = summary.sort_values(
    [
        "moderate_threshold",
        "high_threshold",
    ]
).reset_index(drop=True)

# ---------------------------------------------------------------------
# Gene-level robustness across the complete threshold grid
# ---------------------------------------------------------------------

scenario_tier_columns = [
    column
    for column in tier_matrix.columns
    if column.endswith("__tier")
]

tier_matrix[
    "n_distinct_tiers_across_threshold_grid"
] = (
    tier_matrix[scenario_tier_columns]
    .nunique(axis=1)
)

tier_matrix[
    "n_threshold_scenarios_changing_primary_tier"
] = (
    tier_matrix[scenario_tier_columns]
    .ne(
        tier_matrix["primary_tier"],
        axis=0
    )
    .sum(axis=1)
)

tier_matrix[
    "percent_threshold_scenarios_changing_primary_tier"
] = (
    100.0
    * tier_matrix[
        "n_threshold_scenarios_changing_primary_tier"
    ]
    / len(scenario_tier_columns)
)

tier_matrix[
    "tier_stable_across_all_threshold_scenarios"
] = (
    tier_matrix[
        "n_distinct_tiers_across_threshold_grid"
    ] == 1
)

tier_matrix["always_HIGH_across_grid"] = (
    tier_matrix[scenario_tier_columns]
    .eq("HIGH")
    .all(axis=1)
)

tier_matrix["always_MODERATE_across_grid"] = (
    tier_matrix[scenario_tier_columns]
    .eq("MODERATE")
    .all(axis=1)
)

tier_matrix["always_UNSTABLE_across_grid"] = (
    tier_matrix[scenario_tier_columns]
    .eq("UNSTABLE")
    .all(axis=1)
)

# Distances from each original decision boundary.
tier_matrix[
    "distance_from_MODERATE_threshold_0_45"
] = (
    tier_matrix["MCI"]
    - PRIMARY_MODERATE_THRESHOLD
).abs()

tier_matrix[
    "distance_from_HIGH_threshold_0_70"
] = (
    tier_matrix["MCI"]
    - PRIMARY_HIGH_THRESHOLD
).abs()

tier_matrix[
    "distance_from_nearest_primary_threshold"
] = tier_matrix[
    [
        "distance_from_MODERATE_threshold_0_45",
        "distance_from_HIGH_threshold_0_70",
    ]
].min(axis=1)

# Count how often each gene occupies each tier.
tier_matrix["n_scenarios_HIGH"] = (
    tier_matrix[scenario_tier_columns]
    .eq("HIGH")
    .sum(axis=1)
)

tier_matrix["n_scenarios_MODERATE"] = (
    tier_matrix[scenario_tier_columns]
    .eq("MODERATE")
    .sum(axis=1)
)

tier_matrix["n_scenarios_UNSTABLE"] = (
    tier_matrix[scenario_tier_columns]
    .eq("UNSTABLE")
    .sum(axis=1)
)

tier_matrix = tier_matrix.sort_values(
    [
        "n_threshold_scenarios_changing_primary_tier",
        "distance_from_nearest_primary_threshold",
        "MCI",
    ],
    ascending=[False, True, False]
).reset_index(drop=True)

# ---------------------------------------------------------------------
# Focused one-threshold-at-a-time summaries
# ---------------------------------------------------------------------

moderate_only = summary.loc[
    np.isclose(
        summary["high_threshold"],
        PRIMARY_HIGH_THRESHOLD
    )
].copy()

high_only = summary.loc[
    np.isclose(
        summary["moderate_threshold"],
        PRIMARY_MODERATE_THRESHOLD
    )
].copy()

# ---------------------------------------------------------------------
# Save files
# ---------------------------------------------------------------------

summary_path = (
    OUTPUT_DIR
    / "TASK11_STEP7_THRESHOLD_SENSITIVITY_SUMMARY.csv"
)

gene_level_path = (
    OUTPUT_DIR
    / "TASK11_STEP7_THRESHOLD_SENSITIVITY_GENE_LEVEL.csv"
)

transition_path = (
    OUTPUT_DIR
    / "TASK11_STEP7_THRESHOLD_TIER_TRANSITIONS.csv"
)

tier_matrix_path = (
    OUTPUT_DIR
    / "TASK11_STEP7_THRESHOLD_FULL_TIER_MATRIX.csv"
)

summary.to_csv(summary_path, index=False)
tier_matrix.to_csv(gene_level_path, index=False)
transitions.to_csv(transition_path, index=False)
tier_matrix.to_csv(tier_matrix_path, index=False)

# ---------------------------------------------------------------------
# Print key findings
# ---------------------------------------------------------------------

print("\n" + "=" * 118)
print("MODERATE-THRESHOLD SHIFTS WITH HIGH FIXED AT 0.70")
print("=" * 118)

print(
    moderate_only[
        [
            "moderate_threshold",
            "high_threshold",
            "n_tier_changes",
            "percent_tier_changes",
            "n_promoted_relative_to_primary",
            "n_demoted_relative_to_primary",
            "n_HIGH",
            "n_MODERATE",
            "n_UNSTABLE",
        ]
    ]
    .round(4)
    .to_string(index=False)
)

print("\n" + "=" * 118)
print("HIGH-THRESHOLD SHIFTS WITH MODERATE FIXED AT 0.45")
print("=" * 118)

print(
    high_only[
        [
            "moderate_threshold",
            "high_threshold",
            "n_tier_changes",
            "percent_tier_changes",
            "n_promoted_relative_to_primary",
            "n_demoted_relative_to_primary",
            "n_HIGH",
            "n_MODERATE",
            "n_UNSTABLE",
        ]
    ]
    .round(4)
    .to_string(index=False)
)

print("\n" + "=" * 118)
print("FULL THRESHOLD-GRID ROBUSTNESS")
print("=" * 118)

n_stable_all = int(
    tier_matrix[
        "tier_stable_across_all_threshold_scenarios"
    ].sum()
)

n_always_high = int(
    tier_matrix["always_HIGH_across_grid"].sum()
)

n_always_moderate = int(
    tier_matrix["always_MODERATE_across_grid"].sum()
)

n_always_unstable = int(
    tier_matrix[
        "always_UNSTABLE_across_grid"
    ].sum()
)

print(
    f"Threshold scenarios tested: "
    f"{len(scenario_tier_columns)}"
)

print(
    "Genes retaining one tier across every threshold pair: "
    f"{n_stable_all}/{len(tier_matrix)} "
    f"({100 * n_stable_all / len(tier_matrix):.1f}%)"
)

print(
    f"Always HIGH across the grid: {n_always_high}"
)

print(
    f"Always MODERATE across the grid: "
    f"{n_always_moderate}"
)

print(
    f"Always UNSTABLE across the grid: "
    f"{n_always_unstable}"
)

print("\n" + "=" * 118)
print("GENES MOST SENSITIVE TO THRESHOLD SHIFTS")
print("=" * 118)

sensitive_columns = [
    "gene_symbol",
    "MCI",
    "primary_tier",
    "distance_from_nearest_primary_threshold",
    "n_distinct_tiers_across_threshold_grid",
    "n_threshold_scenarios_changing_primary_tier",
    "percent_threshold_scenarios_changing_primary_tier",
    "n_scenarios_HIGH",
    "n_scenarios_MODERATE",
    "n_scenarios_UNSTABLE",
]

sensitive_genes = tier_matrix.loc[
    tier_matrix[
        "n_threshold_scenarios_changing_primary_tier"
    ] > 0,
    sensitive_columns
].copy()

if len(sensitive_genes) == 0:
    print(
        "No genes changed tier under the tested "
        "threshold shifts."
    )
else:
    print(
        sensitive_genes
        .round(4)
        .to_string(index=False)
    )

print("\n" + "=" * 118)
print("TEN GENES CLOSEST TO A PRIMARY THRESHOLD")
print("=" * 118)

closest = (
    tier_matrix[
        [
            "gene_symbol",
            "MCI",
            "primary_tier",
            "distance_from_MODERATE_threshold_0_45",
            "distance_from_HIGH_threshold_0_70",
            "distance_from_nearest_primary_threshold",
        ]
    ]
    .sort_values(
        "distance_from_nearest_primary_threshold"
    )
    .head(10)
)

print(
    closest
    .round(4)
    .to_string(index=False)
)

print("\n" + "=" * 118)
print("OUTPUT FILES")
print("=" * 118)
print(summary_path)
print(gene_level_path)
print(transition_path)
print(tier_matrix_path)

print("\nSTEP 7 COMPLETE")


In [ ]:
# STEP 8 — Leave-one-cohort-out stability analysis
#
# For each of the three HCM cohorts:
# 1. Remove that cohort.
# 2. Recompute D_g, S_g, R_g, MCI, and tier.
# 3. Compare the reduced-cohort score with the full primary MCI.
#
# Outputs:
# - scenario-level stability summary
# - gene-level leave-one-cohort-out results
# - tier-transition table
# - per-gene robustness summary

from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr

PROJECT_DIR = PROJECT_DIR
MCI_DIR = PROJECT_DIR / "results" / "mci_scores"
OUTPUT_DIR = PROJECT_DIR / "results" / "task11_benchmarking"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

clean_path = (
    MCI_DIR
    / "TASK4_CODE23_HCM_only_clean_DE_input_used_for_MCI.csv"
)

primary_path = (
    MCI_DIR
    / "TASK4_CODE23_REAL_HCM_MCI_scores_primary.csv"
)

clean_de = pd.read_csv(clean_path)
primary = pd.read_csv(primary_path)

eligible = clean_de.loc[
    (clean_de["score_eligible"] == True)
    & (clean_de["matched_in_DE"] == True)
].copy()

for column in ["log2FC", "FDR", "SE", "p_value"]:
    eligible[column] = pd.to_numeric(
        eligible[column],
        errors="coerce"
    )

cohorts = sorted(
    eligible["cohort"]
    .dropna()
    .unique()
    .tolist()
)

print("=" * 120)
print("LEAVE-ONE-COHORT-OUT SETUP")
print("=" * 120)
print(f"Cohorts detected: {cohorts}")
print(f"Primary genes: {primary['gene_symbol'].nunique()}")
print(f"Eligible cohort-level rows: {len(eligible)}")


def calculate_direction_agreement(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) < 2:
        return np.nan

    signs = np.sign(values)

    pair_agreements = [
        int(first == second)
        for first, second in combinations(signs, 2)
    ]

    if len(pair_agreements) == 0:
        return np.nan

    return float(np.mean(pair_agreements))


def calculate_effect_consistency(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) < 2:
        return np.nan

    mean_effect = np.mean(values)
    sd_effect = np.std(values, ddof=1)

    if np.isclose(mean_effect, 0.0):
        return 0.0

    coefficient_of_variation = (
        sd_effect / abs(mean_effect)
    )

    return float(
        1.0
        - min(
            coefficient_of_variation / 2.0,
            1.0
        )
    )


def assign_mci_tier(score, n_cohorts):
    if n_cohorts < 2 or pd.isna(score):
        return "INSUFFICIENT_COVERAGE"

    if score >= 0.70:
        return "HIGH"

    if score >= 0.45:
        return "MODERATE"

    return "UNSTABLE"


def normalize_tier(value):
    if pd.isna(value):
        return "MISSING"

    normalized = (
        str(value)
        .strip()
        .upper()
        .replace("-", "_")
        .replace(" ", "_")
    )

    if "INSUFFICIENT" in normalized:
        return "INSUFFICIENT_COVERAGE"

    return normalized


def inclusive_top_set(data, score_column, k):
    valid = data.loc[
        data[score_column].notna(),
        ["gene_symbol", score_column]
    ].copy()

    valid = valid.sort_values(
        [score_column, "gene_symbol"],
        ascending=[False, True]
    )

    if len(valid) == 0:
        return set(), np.nan

    effective_k = min(k, len(valid))
    cutoff = valid.iloc[
        effective_k - 1
    ][score_column]

    selected = set(
        valid.loc[
            valid[score_column] >= cutoff,
            "gene_symbol"
        ]
    )

    return selected, float(cutoff)


primary["primary_tier_normalized"] = (
    primary["MCI_tier"].apply(normalize_tier)
)

primary_scoreable = primary.loc[
    primary["MCI"].notna()
].copy()

primary_scoreable["primary_rank"] = (
    primary_scoreable["MCI"].rank(
        ascending=False,
        method="average"
    )
)

original_high_genes = set(
    primary_scoreable.loc[
        primary_scoreable[
            "primary_tier_normalized"
        ] == "HIGH",
        "gene_symbol"
    ]
)

n_original_high = len(original_high_genes)

scenario_rows = []
gene_rows = []
transition_rows = []

for held_out_cohort in cohorts:

    remaining = eligible.loc[
        eligible["cohort"] != held_out_cohort
    ].copy()

    recomputed_rows = []

    for gene, group in remaining.groupby(
        "gene_symbol",
        sort=True
    ):
        effects = (
            group["log2FC"]
            .dropna()
            .to_numpy(dtype=float)
        )

        fdr_values = (
            group["FDR"]
            .dropna()
            .to_numpy(dtype=float)
        )

        n_remaining_cohorts = len(effects)

        D_g = calculate_direction_agreement(
            effects
        )

        S_g = calculate_effect_consistency(
            effects
        )

        R_g = (
            float(np.mean(fdr_values < 0.05))
            if len(fdr_values) > 0
            else np.nan
        )

        if (
            n_remaining_cohorts >= 2
            and pd.notna(D_g)
            and pd.notna(S_g)
            and pd.notna(R_g)
        ):
            loo_mci = (
                0.40 * D_g
                + 0.35 * S_g
                + 0.25 * R_g
            )
        else:
            loo_mci = np.nan

        recomputed_rows.append(
            {
                "gene_symbol": gene,
                "held_out_cohort": held_out_cohort,
                "remaining_cohorts": ";".join(
                    sorted(
                        group["cohort"]
                        .astype(str)
                        .unique()
                        .tolist()
                    )
                ),
                "n_remaining_cohorts": (
                    n_remaining_cohorts
                ),
                "loo_D_g": D_g,
                "loo_S_g": S_g,
                "loo_R_g": R_g,
                "loo_MCI": loo_mci,
                "loo_tier": assign_mci_tier(
                    loo_mci,
                    n_remaining_cohorts
                ),
            }
        )

    recomputed = pd.DataFrame(
        recomputed_rows
    )

    scenario = primary_scoreable.merge(
        recomputed,
        on="gene_symbol",
        how="left",
        validate="one_to_one"
    )

    scenario["loo_tier_normalized"] = (
        scenario["loo_tier"].apply(
            normalize_tier
        )
    )

    scenario["loo_rank"] = (
        scenario["loo_MCI"].rank(
            ascending=False,
            method="average"
        )
    )

    scenario["MCI_change"] = (
        scenario["loo_MCI"]
        - scenario["MCI"]
    )

    scenario["absolute_MCI_change"] = (
        scenario["MCI_change"].abs()
    )

    scenario["rank_change"] = (
        scenario["loo_rank"]
        - scenario["primary_rank"]
    )

    scenario["absolute_rank_change"] = (
        scenario["rank_change"].abs()
    )

    scenario["tier_match"] = (
        scenario["primary_tier_normalized"]
        == scenario["loo_tier_normalized"]
    )

    comparable = scenario.loc[
        scenario["MCI"].notna()
        & scenario["loo_MCI"].notna()
    ].copy()

    spearman_result = spearmanr(
        comparable["MCI"],
        comparable["loo_MCI"]
    )

    kendall_result = kendalltau(
        comparable["MCI"],
        comparable["loo_MCI"]
    )

    loo_top_high, loo_cutoff = (
        inclusive_top_set(
            comparable,
            "loo_MCI",
            n_original_high
        )
    )

    high_overlap = len(
        original_high_genes.intersection(
            loo_top_high
        )
    )

    high_union = len(
        original_high_genes.union(
            loo_top_high
        )
    )

    comparable_tier_matches = int(
        comparable["tier_match"].sum()
    )

    comparable_tier_changes = int(
        (~comparable["tier_match"]).sum()
    )

    scenario_rows.append(
        {
            "held_out_cohort": held_out_cohort,
            "remaining_cohorts": ";".join(
                [
                    cohort
                    for cohort in cohorts
                    if cohort != held_out_cohort
                ]
            ),
            "n_primary_scoreable_genes": (
                len(primary_scoreable)
            ),
            "n_genes_with_recomputed_MCI": (
                len(comparable)
            ),
            "n_genes_becoming_insufficient": int(
                scenario["loo_MCI"].isna().sum()
            ),
            "spearman_rho_vs_full_MCI": (
                spearman_result.statistic
            ),
            "spearman_p_value": (
                spearman_result.pvalue
            ),
            "kendall_tau_vs_full_MCI": (
                kendall_result.statistic
            ),
            "kendall_p_value": (
                kendall_result.pvalue
            ),
            "mean_absolute_MCI_change": (
                comparable[
                    "absolute_MCI_change"
                ].mean()
            ),
            "median_absolute_MCI_change": (
                comparable[
                    "absolute_MCI_change"
                ].median()
            ),
            "maximum_absolute_MCI_change": (
                comparable[
                    "absolute_MCI_change"
                ].max()
            ),
            "mean_absolute_rank_change": (
                comparable[
                    "absolute_rank_change"
                ].mean()
            ),
            "median_absolute_rank_change": (
                comparable[
                    "absolute_rank_change"
                ].median()
            ),
            "maximum_absolute_rank_change": (
                comparable[
                    "absolute_rank_change"
                ].max()
            ),
            "exact_tier_matches": (
                comparable_tier_matches
            ),
            "n_tier_changes": (
                comparable_tier_changes
            ),
            "percent_tier_changes": (
                100.0
                * comparable_tier_changes
                / len(comparable)
            ),
            "loo_n_HIGH": int(
                (
                    comparable[
                        "loo_tier_normalized"
                    ] == "HIGH"
                ).sum()
            ),
            "loo_n_MODERATE": int(
                (
                    comparable[
                        "loo_tier_normalized"
                    ] == "MODERATE"
                ).sum()
            ),
            "loo_n_UNSTABLE": int(
                (
                    comparable[
                        "loo_tier_normalized"
                    ] == "UNSTABLE"
                ).sum()
            ),
            "loo_top_n_cutoff": loo_cutoff,
            "loo_top_n_inclusive_size": (
                len(loo_top_high)
            ),
            "original_HIGH_overlap": (
                high_overlap
            ),
            "original_HIGH_recall": (
                high_overlap
                / n_original_high
            ),
            "original_HIGH_jaccard": (
                high_overlap
                / high_union
                if high_union > 0
                else np.nan
            ),
        }
    )

    transition_table = pd.crosstab(
        comparable[
            "primary_tier_normalized"
        ],
        comparable[
            "loo_tier_normalized"
        ],
        rownames=["primary_tier"],
        colnames=["leave_one_out_tier"]
    ).reindex(
        index=[
            "HIGH",
            "MODERATE",
            "UNSTABLE",
        ],
        columns=[
            "HIGH",
            "MODERATE",
            "UNSTABLE",
        ],
        fill_value=0
    )

    for primary_tier in [
        "HIGH",
        "MODERATE",
        "UNSTABLE",
    ]:
        for loo_tier in [
            "HIGH",
            "MODERATE",
            "UNSTABLE",
        ]:
            transition_rows.append(
                {
                    "held_out_cohort": (
                        held_out_cohort
                    ),
                    "primary_tier": (
                        primary_tier
                    ),
                    "leave_one_out_tier": (
                        loo_tier
                    ),
                    "n_genes": int(
                        transition_table.loc[
                            primary_tier,
                            loo_tier
                        ]
                    ),
                }
            )

    selected_gene_columns = [
        "gene_symbol",
        "MCI",
        "primary_tier_normalized",
        "primary_rank",
        "held_out_cohort",
        "remaining_cohorts",
        "n_remaining_cohorts",
        "loo_D_g",
        "loo_S_g",
        "loo_R_g",
        "loo_MCI",
        "loo_tier_normalized",
        "loo_rank",
        "MCI_change",
        "absolute_MCI_change",
        "rank_change",
        "absolute_rank_change",
        "tier_match",
    ]

    gene_rows.append(
        scenario[selected_gene_columns]
    )

scenario_summary = pd.DataFrame(
    scenario_rows
)

gene_level = pd.concat(
    gene_rows,
    ignore_index=True
)

tier_transitions = pd.DataFrame(
    transition_rows
)

# ---------------------------------------------------------------------
# Per-gene robustness across the three held-out scenarios
# ---------------------------------------------------------------------

gene_robustness_rows = []

for gene, group in gene_level.groupby(
    "gene_symbol",
    sort=True
):
    primary_mci = group["MCI"].iloc[0]
    primary_tier = group[
        "primary_tier_normalized"
    ].iloc[0]

    valid = group.loc[
        group["loo_MCI"].notna()
    ].copy()

    gene_robustness_rows.append(
        {
            "gene_symbol": gene,
            "primary_MCI": primary_mci,
            "primary_tier": primary_tier,
            "n_leave_one_out_scenarios": (
                len(group)
            ),
            "n_valid_leave_one_out_scores": (
                len(valid)
            ),
            "minimum_leave_one_out_MCI": (
                valid["loo_MCI"].min()
                if len(valid) > 0
                else np.nan
            ),
            "maximum_leave_one_out_MCI": (
                valid["loo_MCI"].max()
                if len(valid) > 0
                else np.nan
            ),
            "leave_one_out_MCI_range": (
                valid["loo_MCI"].max()
                - valid["loo_MCI"].min()
                if len(valid) > 0
                else np.nan
            ),
            "mean_absolute_MCI_change": (
                valid[
                    "absolute_MCI_change"
                ].mean()
                if len(valid) > 0
                else np.nan
            ),
            "maximum_absolute_MCI_change": (
                valid[
                    "absolute_MCI_change"
                ].max()
                if len(valid) > 0
                else np.nan
            ),
            "n_leave_one_out_tier_changes": int(
                (~valid["tier_match"]).sum()
            ),
            "tier_stable_across_all_valid_leave_one_out_scenarios": (
                bool(valid["tier_match"].all())
                if len(valid) > 0
                else False
            ),
            "n_scenarios_becoming_insufficient": int(
                group["loo_MCI"].isna().sum()
            ),
        }
    )

gene_robustness = pd.DataFrame(
    gene_robustness_rows
)

gene_robustness = gene_robustness.sort_values(
    [
        "n_leave_one_out_tier_changes",
        "maximum_absolute_MCI_change",
    ],
    ascending=[False, False]
).reset_index(drop=True)

# ---------------------------------------------------------------------
# Save outputs
# ---------------------------------------------------------------------

summary_path = (
    OUTPUT_DIR
    / "TASK11_STEP8_LEAVE_ONE_COHORT_OUT_SUMMARY.csv"
)

gene_level_path = (
    OUTPUT_DIR
    / "TASK11_STEP8_LEAVE_ONE_COHORT_OUT_GENE_LEVEL.csv"
)

transition_path = (
    OUTPUT_DIR
    / "TASK11_STEP8_LEAVE_ONE_COHORT_OUT_TIER_TRANSITIONS.csv"
)

gene_robustness_path = (
    OUTPUT_DIR
    / "TASK11_STEP8_LEAVE_ONE_COHORT_OUT_GENE_ROBUSTNESS.csv"
)

scenario_summary.to_csv(
    summary_path,
    index=False
)

gene_level.to_csv(
    gene_level_path,
    index=False
)

tier_transitions.to_csv(
    transition_path,
    index=False
)

gene_robustness.to_csv(
    gene_robustness_path,
    index=False
)

# ---------------------------------------------------------------------
# Print key results
# ---------------------------------------------------------------------

print("\n" + "=" * 120)
print("LEAVE-ONE-COHORT-OUT STABILITY SUMMARY")
print("=" * 120)

summary_display_columns = [
    "held_out_cohort",
    "n_genes_with_recomputed_MCI",
    "n_genes_becoming_insufficient",
    "spearman_rho_vs_full_MCI",
    "kendall_tau_vs_full_MCI",
    "mean_absolute_MCI_change",
    "maximum_absolute_MCI_change",
    "mean_absolute_rank_change",
    "maximum_absolute_rank_change",
    "n_tier_changes",
    "percent_tier_changes",
    "loo_n_HIGH",
    "loo_n_MODERATE",
    "loo_n_UNSTABLE",
    "original_HIGH_overlap",
    "original_HIGH_recall",
]

print(
    scenario_summary[
        summary_display_columns
    ]
    .round(4)
    .to_string(index=False)
)

print("\n" + "=" * 120)
print("OVERALL GENE-LEVEL LEAVE-ONE-COHORT-OUT ROBUSTNESS")
print("=" * 120)

fully_stable = gene_robustness.loc[
    (
        gene_robustness[
            "tier_stable_across_all_valid_leave_one_out_scenarios"
        ] == True
    )
    & (
        gene_robustness[
            "n_scenarios_becoming_insufficient"
        ] == 0
    )
]

any_tier_change = gene_robustness.loc[
    gene_robustness[
        "n_leave_one_out_tier_changes"
    ] > 0
]

became_insufficient = gene_robustness.loc[
    gene_robustness[
        "n_scenarios_becoming_insufficient"
    ] > 0
]

print(f"Genes assessed: {len(gene_robustness)}")

print(
    "Genes retaining the same tier in all three "
    "leave-one-cohort-out scenarios: "
    f"{len(fully_stable)}/{len(gene_robustness)} "
    f"({100 * len(fully_stable) / len(gene_robustness):.1f}%)"
)

print(
    "Genes changing tier in at least one valid scenario: "
    f"{len(any_tier_change)}/{len(gene_robustness)} "
    f"({100 * len(any_tier_change) / len(gene_robustness):.1f}%)"
)

print(
    "Genes becoming insufficient after removal of at least one cohort: "
    f"{len(became_insufficient)}/{len(gene_robustness)}"
)

print("\n" + "=" * 120)
print("GENES WITH LEAVE-ONE-COHORT-OUT TIER CHANGES")
print("=" * 120)

changed_columns = [
    "gene_symbol",
    "primary_MCI",
    "primary_tier",
    "minimum_leave_one_out_MCI",
    "maximum_leave_one_out_MCI",
    "leave_one_out_MCI_range",
    "maximum_absolute_MCI_change",
    "n_leave_one_out_tier_changes",
    "n_scenarios_becoming_insufficient",
]

changed_genes = gene_robustness.loc[
    (
        gene_robustness[
            "n_leave_one_out_tier_changes"
        ] > 0
    )
    | (
        gene_robustness[
            "n_scenarios_becoming_insufficient"
        ] > 0
    ),
    changed_columns
]

if len(changed_genes) == 0:
    print("No genes changed tier or became insufficient.")
else:
    print(
        changed_genes
        .round(4)
        .to_string(index=False)
    )

print("\n" + "=" * 120)
print("OUTPUT FILES")
print("=" * 120)
print(summary_path)
print(gene_level_path)
print(transition_path)
print(gene_robustness_path)

print("\nSTEP 8 COMPLETE")


In [ ]:
# STEP 9 — External-evidence benchmarking
#
# Compares the original MCI and five alternative methods against:
# 1. Held-out strict expression validation
# 2. Held-out directional expression validation
# 3. GWAS Catalog HCM convergence
#
# Metrics:
# - AUROC
# - Average precision
# - One-sided Mann-Whitney test
# - Top-18 enrichment
# - Fisher exact test
#
# Top sets include all ties at the 18th-ranked score.
# This is especially important for direction voting, which has many ties.

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import fisher_exact, mannwhitneyu
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

if "PROJECT_DIR" not in globals():
    raise RuntimeError(
        "Run Step 1 before executing this analysis cell."
    )

PROJECT_DIR = Path(PROJECT_DIR)

MCI_DIR = (
    PROJECT_DIR
    / "results"
    / "mci_scores"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "results"
    / "task11_benchmarking"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

benchmark_path = (
    OUTPUT_DIR
    / "TASK11_STEP4_METHOD_BENCHMARK_GENE_LEVEL.csv"
)

heldout_path = (
    MCI_DIR
    / "TASK9_PATHA_HELDOUT_GSE160997_VALIDATION_GENE_LEVEL.csv"
)

gwas_path = (
    MCI_DIR
    / "TASK9_PATHB_GWAS_CATALOG_CONVERGENCE_GENE_LEVEL.csv"
)

benchmark = pd.read_csv(benchmark_path)
heldout = pd.read_csv(heldout_path)
gwas = pd.read_csv(gwas_path)


def normalize_boolean(value):
    """
    Convert common CSV boolean representations into True, False, or NaN.
    """
    if pd.isna(value):
        return np.nan

    if isinstance(value, bool):
        return value

    if isinstance(value, (int, np.integer)):
        if value == 1:
            return True
        if value == 0:
            return False

    if isinstance(value, (float, np.floating)):
        if np.isclose(value, 1.0):
            return True
        if np.isclose(value, 0.0):
            return False

    normalized = str(value).strip().lower()

    true_values = {
        "true",
        "t",
        "yes",
        "y",
        "1",
    }

    false_values = {
        "false",
        "f",
        "no",
        "n",
        "0",
    }

    if normalized in true_values:
        return True

    if normalized in false_values:
        return False

    return np.nan


boolean_columns_heldout = [
    "heldout_matched",
    "heldout_validated_strict",
    "heldout_validated_directional",
]

for column in boolean_columns_heldout:
    heldout[column] = heldout[column].apply(
        normalize_boolean
    )

boolean_columns_gwas = [
    "catalog_hcm_gwas_match",
    "gwas_matched",
    "gwas_nominal_p_lt_0_05",
    "gwas_FDR_lt_0_05",
    "gwas_genomewide_p_lt_5e_8",
]

for column in boolean_columns_gwas:
    gwas[column] = gwas[column].apply(
        normalize_boolean
    )


# ---------------------------------------------------------------------
# Create common scoreable analysis set
# ---------------------------------------------------------------------

method_columns = {
    "Original MCI": "MCI",
    "Equal-weight composite": (
        "equal_weight_composite"
    ),
    "Direction vote": (
        "direction_vote_score"
    ),
    "Fisher combined significance": (
        "fisher_minus_log10_p"
    ),
    "Signed Stouffer significance": (
        "stouffer_minus_log10_p"
    ),
    "Random-effects meta-analysis": (
        "random_effect_minus_log10_p"
    ),
}

scoreable = benchmark.loc[
    benchmark["MCI"].notna()
].copy()

heldout_fields = heldout[
    [
        "gene_symbol",
        "heldout_matched",
        "heldout_validated_strict",
        "heldout_validated_directional",
        "heldout_same_direction_as_primary",
        "heldout_FDR_lt_0_05",
        "heldout_nominal_p_lt_0_05",
        "heldout_log2FC",
        "heldout_FDR",
        "heldout_p_value",
    ]
].copy()

gwas_fields = gwas[
    [
        "gene_symbol",
        "catalog_hcm_gwas_match",
        "gwas_matched",
        "gwas_nominal_p_lt_0_05",
        "gwas_FDR_lt_0_05",
        "gwas_genomewide_p_lt_5e_8",
        "gwas_p_value",
        "minus_log10_gwas_p",
        "gwas_top_decile_signal",
    ]
].copy()

external = (
    scoreable
    .merge(
        heldout_fields,
        on="gene_symbol",
        how="left",
        validate="one_to_one"
    )
    .merge(
        gwas_fields,
        on="gene_symbol",
        how="left",
        validate="one_to_one"
    )
)

print("=" * 120)
print("EXTERNAL-EVIDENCE ANALYSIS SET")
print("=" * 120)

print(f"Scoreable genes: {len(external)}")

print(
    "Held-out matched genes: "
    f"{external['heldout_matched'].eq(True).sum()}"
)

print(
    "Held-out strict validations among matched genes: "
    f"{external.loc[external['heldout_matched'].eq(True), 'heldout_validated_strict'].eq(True).sum()}"
)

print(
    "Held-out directional validations among matched genes: "
    f"{external.loc[external['heldout_matched'].eq(True), 'heldout_validated_directional'].eq(True).sum()}"
)

print(
    "GWAS Catalog matches among scoreable genes: "
    f"{external['catalog_hcm_gwas_match'].eq(True).sum()}"
)


# ---------------------------------------------------------------------
# Construct inclusive top-18 membership for each method
# ---------------------------------------------------------------------

TOP_N = 18

top_set_metadata = []

for method_name, score_column in method_columns.items():

    valid_scores = external.loc[
        external[score_column].notna(),
        [
            "gene_symbol",
            score_column,
        ]
    ].copy()

    valid_scores = valid_scores.sort_values(
        [
            score_column,
            "gene_symbol",
        ],
        ascending=[
            False,
            True,
        ]
    )

    effective_n = min(
        TOP_N,
        len(valid_scores)
    )

    cutoff = valid_scores.iloc[
        effective_n - 1
    ][score_column]

    top_genes = set(
        valid_scores.loc[
            valid_scores[score_column] >= cutoff,
            "gene_symbol"
        ]
    )

    membership_column = (
        "top18__"
        + method_name
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    external[membership_column] = (
        external["gene_symbol"].isin(
            top_genes
        )
    )

    number_tied_at_cutoff = int(
        np.isclose(
            valid_scores[score_column],
            cutoff,
            rtol=1e-12,
            atol=1e-12
        ).sum()
    )

    top_set_metadata.append(
        {
            "method": method_name,
            "score_column": score_column,
            "membership_column": (
                membership_column
            ),
            "target_top_n": TOP_N,
            "top_score_cutoff": cutoff,
            "inclusive_top_set_size": (
                len(top_genes)
            ),
            "n_genes_tied_at_cutoff": (
                number_tied_at_cutoff
            ),
        }
    )

top_set_metadata = pd.DataFrame(
    top_set_metadata
)


# ---------------------------------------------------------------------
# Define external outcomes
# ---------------------------------------------------------------------

outcome_definitions = {
    "Held-out strict validation": {
        "outcome_column": (
            "heldout_validated_strict"
        ),
        "eligibility_column": (
            "heldout_matched"
        ),
        "eligibility_value": True,
    },

    "Held-out directional validation": {
        "outcome_column": (
            "heldout_validated_directional"
        ),
        "eligibility_column": (
            "heldout_matched"
        ),
        "eligibility_value": True,
    },

    "GWAS Catalog HCM match": {
        "outcome_column": (
            "catalog_hcm_gwas_match"
        ),
        "eligibility_column": None,
        "eligibility_value": None,
    },
}


def calculate_auc_metrics(
    outcomes,
    scores
):
    """
    Calculate AUROC and average precision when both classes are present.
    """
    outcomes = np.asarray(
        outcomes,
        dtype=int
    )

    scores = np.asarray(
        scores,
        dtype=float
    )

    if len(np.unique(outcomes)) < 2:
        return np.nan, np.nan

    auroc = roc_auc_score(
        outcomes,
        scores
    )

    average_precision = (
        average_precision_score(
            outcomes,
            scores
        )
    )

    return (
        float(auroc),
        float(average_precision),
    )


def safe_enrichment_ratio(
    top_rate,
    rest_rate
):
    if pd.isna(top_rate) or pd.isna(rest_rate):
        return np.nan

    if np.isclose(rest_rate, 0.0):
        if top_rate > 0:
            return np.inf
        return np.nan

    return float(
        top_rate / rest_rate
    )


summary_rows = []

for outcome_name, outcome_definition in outcome_definitions.items():

    outcome_column = outcome_definition[
        "outcome_column"
    ]

    eligibility_column = outcome_definition[
        "eligibility_column"
    ]

    if eligibility_column is None:
        outcome_data = external.copy()
    else:
        outcome_data = external.loc[
            external[
                eligibility_column
            ].eq(
                outcome_definition[
                    "eligibility_value"
                ]
            )
        ].copy()

    outcome_data = outcome_data.loc[
        outcome_data[outcome_column].notna()
    ].copy()

    outcome_data[
        "_binary_outcome"
    ] = (
        outcome_data[outcome_column]
        .eq(True)
        .astype(int)
    )

    for _, metadata in top_set_metadata.iterrows():

        method_name = metadata["method"]
        score_column = metadata[
            "score_column"
        ]

        membership_column = metadata[
            "membership_column"
        ]

        valid = outcome_data.loc[
            outcome_data[score_column].notna()
        ].copy()

        n_total = len(valid)
        n_positive = int(
            valid["_binary_outcome"].sum()
        )

        n_negative = int(
            n_total - n_positive
        )

        prevalence = (
            n_positive / n_total
            if n_total > 0
            else np.nan
        )

        auroc, average_precision = (
            calculate_auc_metrics(
                valid["_binary_outcome"],
                valid[score_column]
            )
        )

        positive_scores = valid.loc[
            valid["_binary_outcome"] == 1,
            score_column
        ]

        negative_scores = valid.loc[
            valid["_binary_outcome"] == 0,
            score_column
        ]

        if (
            len(positive_scores) > 0
            and len(negative_scores) > 0
        ):
            mann_whitney = mannwhitneyu(
                positive_scores,
                negative_scores,
                alternative="greater",
                method="auto"
            )

            mann_whitney_u = float(
                mann_whitney.statistic
            )

            mann_whitney_p = float(
                mann_whitney.pvalue
            )
        else:
            mann_whitney_u = np.nan
            mann_whitney_p = np.nan

        top_group = valid.loc[
            valid[membership_column]
        ]

        rest_group = valid.loc[
            ~valid[membership_column]
        ]

        top_positive = int(
            top_group[
                "_binary_outcome"
            ].sum()
        )

        top_negative = int(
            len(top_group) - top_positive
        )

        rest_positive = int(
            rest_group[
                "_binary_outcome"
            ].sum()
        )

        rest_negative = int(
            len(rest_group) - rest_positive
        )

        top_rate = (
            top_positive / len(top_group)
            if len(top_group) > 0
            else np.nan
        )

        rest_rate = (
            rest_positive / len(rest_group)
            if len(rest_group) > 0
            else np.nan
        )

        if (
            len(top_group) > 0
            and len(rest_group) > 0
        ):
            fisher_result = fisher_exact(
                [
                    [
                        top_positive,
                        top_negative,
                    ],
                    [
                        rest_positive,
                        rest_negative,
                    ],
                ],
                alternative="greater"
            )

            fisher_odds_ratio = float(
                fisher_result.statistic
            )

            fisher_p_value = float(
                fisher_result.pvalue
            )
        else:
            fisher_odds_ratio = np.nan
            fisher_p_value = np.nan

        summary_rows.append(
            {
                "external_outcome": outcome_name,
                "method": method_name,
                "score_column": score_column,
                "n_evaluable_genes": n_total,
                "n_outcome_positive": n_positive,
                "n_outcome_negative": n_negative,
                "outcome_prevalence": prevalence,
                "AUROC": auroc,
                "average_precision": (
                    average_precision
                ),
                "mann_whitney_U": (
                    mann_whitney_u
                ),
                "mann_whitney_one_sided_p": (
                    mann_whitney_p
                ),
                "target_top_n": TOP_N,
                "inclusive_top_set_size_all_scoreable": int(
                    metadata[
                        "inclusive_top_set_size"
                    ]
                ),
                "n_genes_tied_at_top_cutoff": int(
                    metadata[
                        "n_genes_tied_at_cutoff"
                    ]
                ),
                "top_score_cutoff": (
                    metadata[
                        "top_score_cutoff"
                    ]
                ),
                "n_top_evaluable": len(
                    top_group
                ),
                "n_rest_evaluable": len(
                    rest_group
                ),
                "top_outcome_positive": (
                    top_positive
                ),
                "top_outcome_negative": (
                    top_negative
                ),
                "rest_outcome_positive": (
                    rest_positive
                ),
                "rest_outcome_negative": (
                    rest_negative
                ),
                "top_positive_rate": top_rate,
                "rest_positive_rate": rest_rate,
                "top_vs_rest_enrichment_ratio": (
                    safe_enrichment_ratio(
                        top_rate,
                        rest_rate
                    )
                ),
                "fisher_exact_odds_ratio": (
                    fisher_odds_ratio
                ),
                "fisher_exact_one_sided_p": (
                    fisher_p_value
                ),
            }
        )

summary = pd.DataFrame(
    summary_rows
)

summary["outcome_order"] = summary[
    "external_outcome"
].map(
    {
        "Held-out strict validation": 1,
        "Held-out directional validation": 2,
        "GWAS Catalog HCM match": 3,
    }
)

summary["method_order"] = summary[
    "method"
].map(
    {
        method: index
        for index, method in enumerate(
            method_columns.keys(),
            start=1
        )
    }
)

summary = (
    summary
    .sort_values(
        [
            "outcome_order",
            "method_order",
        ]
    )
    .drop(
        columns=[
            "outcome_order",
            "method_order",
        ]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------
# Save outputs
# ---------------------------------------------------------------------

summary_path = (
    OUTPUT_DIR
    / "TASK11_STEP9_EXTERNAL_EVIDENCE_BENCHMARK_SUMMARY.csv"
)

gene_level_path = (
    OUTPUT_DIR
    / "TASK11_STEP9_EXTERNAL_EVIDENCE_GENE_LEVEL.csv"
)

top_set_path = (
    OUTPUT_DIR
    / "TASK11_STEP9_METHOD_TOP_SET_METADATA.csv"
)

summary.to_csv(
    summary_path,
    index=False
)

external.to_csv(
    gene_level_path,
    index=False
)

top_set_metadata.to_csv(
    top_set_path,
    index=False
)


# ---------------------------------------------------------------------
# Print key results
# ---------------------------------------------------------------------

for outcome_name in outcome_definitions.keys():

    outcome_summary = summary.loc[
        summary[
            "external_outcome"
        ] == outcome_name
    ].copy()

    print("\n" + "=" * 120)
    print(outcome_name.upper())
    print("=" * 120)

    display_columns = [
        "method",
        "n_evaluable_genes",
        "n_outcome_positive",
        "AUROC",
        "average_precision",
        "mann_whitney_one_sided_p",
        "inclusive_top_set_size_all_scoreable",
        "n_genes_tied_at_top_cutoff",
        "n_top_evaluable",
        "top_outcome_positive",
        "top_positive_rate",
        "n_rest_evaluable",
        "rest_outcome_positive",
        "rest_positive_rate",
        "top_vs_rest_enrichment_ratio",
        "fisher_exact_odds_ratio",
        "fisher_exact_one_sided_p",
    ]

    print(
        outcome_summary[
            display_columns
        ]
        .round(4)
        .to_string(index=False)
    )

print("\n" + "=" * 120)
print("METHOD TOP-SET CHARACTERISTICS")
print("=" * 120)

print(
    top_set_metadata[
        [
            "method",
            "target_top_n",
            "inclusive_top_set_size",
            "n_genes_tied_at_cutoff",
            "top_score_cutoff",
        ]
    ]
    .round(6)
    .to_string(index=False)
)

print("\n" + "=" * 120)
print("IMPORTANT INTERPRETATION WARNING")
print("=" * 120)

strict_positive_count = int(
    external.loc[
        external["heldout_matched"].eq(True),
        "heldout_validated_strict"
    ].eq(True).sum()
)

print(
    "The strict held-out outcome contains only "
    f"{strict_positive_count} positive genes. "
    "AUROC, average precision, enrichment ratios, "
    "and p-values must therefore be interpreted as "
    "exploratory rather than definitive."
)

print("\n" + "=" * 120)
print("OUTPUT FILES")
print("=" * 120)
print(summary_path)
print(gene_level_path)
print(top_set_path)

print("\nSTEP 9 COMPLETE")


In [ ]:
# STEP 10 — Create publication-ready benchmarking tables and figures
#
# Creates:
# 1. Compact cross-method benchmarking table
# 2. Full cross-method benchmarking table
# 3. Sensitivity-analysis summary tables
# 4. High-resolution PNG, PDF, and SVG figures
#
# No statistical analyses are changed in this step.
# This step only consolidates and visualizes the completed results.

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "PROJECT_DIR" not in globals():
    raise RuntimeError(
        "Run Step 1 before executing this analysis cell."
    )

PROJECT_DIR = Path(PROJECT_DIR)

TASK11_DIR = (
    PROJECT_DIR
    / "results"
    / "task11_benchmarking"
)

PUBLICATION_DIR = (
    TASK11_DIR
    / "publication_ready"
)

TABLE_DIR = (
    PUBLICATION_DIR
    / "tables"
)

FIGURE_DIR = (
    PUBLICATION_DIR
    / "figures"
)

TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# -------------------------------------------------------------------
# Load completed Task 11 outputs
# -------------------------------------------------------------------

benchmark = pd.read_csv(
    TASK11_DIR
    / "TASK11_STEP4_METHOD_BENCHMARK_GENE_LEVEL.csv"
)

ranking = pd.read_csv(
    TASK11_DIR
    / "TASK11_STEP5_RANKING_AGREEMENT_SUMMARY.csv"
)

weight_sensitivity = pd.read_csv(
    TASK11_DIR
    / "TASK11_STEP6_WEIGHT_SENSITIVITY_SUMMARY.csv"
)

threshold_sensitivity = pd.read_csv(
    TASK11_DIR
    / "TASK11_STEP7_THRESHOLD_SENSITIVITY_SUMMARY.csv"
)

loo_sensitivity = pd.read_csv(
    TASK11_DIR
    / "TASK11_STEP8_LEAVE_ONE_COHORT_OUT_SUMMARY.csv"
)

external = pd.read_csv(
    TASK11_DIR
    / "TASK11_STEP9_EXTERNAL_EVIDENCE_BENCHMARK_SUMMARY.csv"
)

method_order = [
    "Original MCI",
    "Equal-weight composite",
    "Direction vote",
    "Fisher combined significance",
    "Signed Stouffer significance",
    "Random-effects meta-analysis",
]

score_columns = {
    "Original MCI": "MCI",
    "Equal-weight composite": "equal_weight_composite",
    "Direction vote": "direction_vote_score",
    "Fisher combined significance": "fisher_minus_log10_p",
    "Signed Stouffer significance": "stouffer_minus_log10_p",
    "Random-effects meta-analysis": "random_effect_minus_log10_p",
}

# -------------------------------------------------------------------
# Construct the cross-method ranking table
# -------------------------------------------------------------------

baseline_row = {
    "method": "Original MCI",
    "n_genes_compared": int(
        benchmark["MCI"].notna().sum()
    ),
    "unique_score_values": int(
        benchmark["MCI"].nunique(dropna=True)
    ),
    "largest_tie_size": int(
        benchmark.loc[
            benchmark["MCI"].notna()
        ]
        .groupby("MCI")
        .size()
        .max()
    ),
    "spearman_rho_vs_MCI": 1.0,
    "spearman_p_value": np.nan,
    "kendall_tau_vs_MCI": 1.0,
    "kendall_p_value": np.nan,
    "mean_absolute_rank_shift": 0.0,
    "median_absolute_rank_shift": 0.0,
    "maximum_absolute_rank_shift": 0.0,
    "method_inclusive_top10_size": 10,
    "top10_overlap_with_MCI": 10,
    "top10_original_recall": 1.0,
    "top10_jaccard": 1.0,
    "method_top_high_inclusive_size": 18,
    "original_HIGH_overlap": 18,
    "original_HIGH_recall": 1.0,
    "original_HIGH_jaccard": 1.0,
}

ranking_complete = pd.concat(
    [
        pd.DataFrame([baseline_row]),
        ranking,
    ],
    ignore_index=True,
    sort=False
)

ranking_complete["method_order"] = (
    ranking_complete["method"].map(
        {
            method: index
            for index, method in enumerate(
                method_order
            )
        }
    )
)

ranking_complete = (
    ranking_complete
    .sort_values("method_order")
    .drop(columns="method_order")
    .reset_index(drop=True)
)

# -------------------------------------------------------------------
# Pivot external evidence results into one row per method
# -------------------------------------------------------------------

external_metrics = [
    "AUROC",
    "average_precision",
    "mann_whitney_one_sided_p",
    "top_positive_rate",
    "rest_positive_rate",
    "top_vs_rest_enrichment_ratio",
    "fisher_exact_odds_ratio",
    "fisher_exact_one_sided_p",
    "n_outcome_positive",
    "n_evaluable_genes",
]

external_name_map = {
    "Held-out strict validation": "heldout_strict",
    "Held-out directional validation": "heldout_directional",
    "GWAS Catalog HCM match": "gwas",
}

external_wide_parts = []

for outcome_name, prefix in external_name_map.items():

    part = external.loc[
        external["external_outcome"] == outcome_name,
        ["method"] + external_metrics
    ].copy()

    part = part.rename(
        columns={
            metric: f"{prefix}__{metric}"
            for metric in external_metrics
        }
    )

    external_wide_parts.append(part)

external_wide = external_wide_parts[0]

for part in external_wide_parts[1:]:
    external_wide = external_wide.merge(
        part,
        on="method",
        how="outer",
        validate="one_to_one"
    )

method_full = ranking_complete.merge(
    external_wide,
    on="method",
    how="left",
    validate="one_to_one"
)

method_full["method_order"] = (
    method_full["method"].map(
        {
            method: index
            for index, method in enumerate(
                method_order
            )
        }
    )
)

method_full = (
    method_full
    .sort_values("method_order")
    .drop(columns="method_order")
    .reset_index(drop=True)
)

# Compact manuscript-facing version
compact_columns = [
    "method",
    "spearman_rho_vs_MCI",
    "kendall_tau_vs_MCI",
    "mean_absolute_rank_shift",
    "top10_overlap_with_MCI",
    "original_HIGH_overlap",
    "original_HIGH_recall",
    "heldout_strict__AUROC",
    "heldout_strict__average_precision",
    "heldout_directional__AUROC",
    "gwas__AUROC",
    "gwas__top_positive_rate",
    "gwas__rest_positive_rate",
    "gwas__fisher_exact_one_sided_p",
]

method_compact = method_full[
    compact_columns
].copy()

method_compact = method_compact.rename(
    columns={
        "method": "Method",
        "spearman_rho_vs_MCI": "Spearman_rho_vs_MCI",
        "kendall_tau_vs_MCI": "Kendall_tau_vs_MCI",
        "mean_absolute_rank_shift": "Mean_absolute_rank_shift",
        "top10_overlap_with_MCI": "Top10_overlap_with_MCI",
        "original_HIGH_overlap": "Original_HIGH_overlap_of_18",
        "original_HIGH_recall": "Original_HIGH_recall",
        "heldout_strict__AUROC": "Heldout_strict_AUROC",
        "heldout_strict__average_precision": "Heldout_strict_average_precision",
        "heldout_directional__AUROC": "Heldout_directional_AUROC",
        "gwas__AUROC": "GWAS_AUROC",
        "gwas__top_positive_rate": "GWAS_top_set_match_rate",
        "gwas__rest_positive_rate": "GWAS_rest_match_rate",
        "gwas__fisher_exact_one_sided_p": "GWAS_Fisher_one_sided_p",
    }
)

# -------------------------------------------------------------------
# Consolidated sensitivity table
# -------------------------------------------------------------------

weight_table = weight_sensitivity.copy()

weight_table["analysis_family"] = (
    "Weight sensitivity"
)

weight_table["perturbation"] = (
    weight_table["scenario"]
)

weight_table["rank_agreement"] = (
    weight_table["spearman_rho_vs_primary"]
)

weight_table["tier_changes_n"] = (
    weight_table["n_tier_changes"]
)

weight_table["tier_changes_percent"] = (
    weight_table["percent_tier_changes"]
)

weight_table["genes_evaluable"] = (
    weight_table["n_genes"]
)

weight_table["high_recall"] = (
    weight_table["original_HIGH_recall"]
)

weight_selected = weight_table[
    [
        "analysis_family",
        "perturbation",
        "scenario_type",
        "genes_evaluable",
        "rank_agreement",
        "tier_changes_n",
        "tier_changes_percent",
        "high_recall",
    ]
].copy()

threshold_table = threshold_sensitivity.copy()

threshold_table["analysis_family"] = (
    "Threshold sensitivity"
)

threshold_table["perturbation"] = (
    "MODERATE="
    + threshold_table[
        "moderate_threshold"
    ].map(lambda value: f"{value:.3f}")
    + "; HIGH="
    + threshold_table[
        "high_threshold"
    ].map(lambda value: f"{value:.3f}")
)

threshold_table["scenario_type"] = (
    np.where(
        threshold_table[
            "is_primary_threshold_pair"
        ],
        "Primary",
        "Threshold-grid alternative"
    )
)

threshold_table["genes_evaluable"] = (
    threshold_table["n_genes"]
)

threshold_table["rank_agreement"] = 1.0

threshold_table["tier_changes_n"] = (
    threshold_table["n_tier_changes"]
)

threshold_table["tier_changes_percent"] = (
    threshold_table[
        "percent_tier_changes"
    ]
)

threshold_table["high_recall"] = np.nan

threshold_selected = threshold_table[
    [
        "analysis_family",
        "perturbation",
        "scenario_type",
        "genes_evaluable",
        "rank_agreement",
        "tier_changes_n",
        "tier_changes_percent",
        "high_recall",
    ]
].copy()

loo_table = loo_sensitivity.copy()

loo_table["analysis_family"] = (
    "Leave-one-cohort-out"
)

loo_table["perturbation"] = (
    "Remove " + loo_table["held_out_cohort"]
)

loo_table["scenario_type"] = (
    "Cohort-removal audit"
)

loo_table["genes_evaluable"] = (
    loo_table[
        "n_genes_with_recomputed_MCI"
    ]
)

loo_table["rank_agreement"] = (
    loo_table[
        "spearman_rho_vs_full_MCI"
    ]
)

loo_table["tier_changes_n"] = (
    loo_table["n_tier_changes"]
)

loo_table["tier_changes_percent"] = (
    loo_table["percent_tier_changes"]
)

loo_table["high_recall"] = (
    loo_table["original_HIGH_recall"]
)

loo_selected = loo_table[
    [
        "analysis_family",
        "perturbation",
        "scenario_type",
        "genes_evaluable",
        "rank_agreement",
        "tier_changes_n",
        "tier_changes_percent",
        "high_recall",
    ]
].copy()

sensitivity_all = pd.concat(
    [
        weight_selected,
        threshold_selected,
        loo_selected,
    ],
    ignore_index=True
)

# -------------------------------------------------------------------
# Save publication-ready tables
# -------------------------------------------------------------------

method_full_path = (
    TABLE_DIR
    / "TASK11_TABLE_METHOD_BENCHMARK_FULL.csv"
)

method_compact_path = (
    TABLE_DIR
    / "TASK11_TABLE_METHOD_BENCHMARK_COMPACT.csv"
)

sensitivity_path = (
    TABLE_DIR
    / "TASK11_TABLE_SENSITIVITY_ALL_SCENARIOS.csv"
)

weight_path = (
    TABLE_DIR
    / "TASK11_TABLE_WEIGHT_SENSITIVITY.csv"
)

threshold_path = (
    TABLE_DIR
    / "TASK11_TABLE_THRESHOLD_SENSITIVITY.csv"
)

loo_path = (
    TABLE_DIR
    / "TASK11_TABLE_LEAVE_ONE_COHORT_OUT.csv"
)

method_full.to_csv(
    method_full_path,
    index=False
)

method_compact.to_csv(
    method_compact_path,
    index=False
)

sensitivity_all.to_csv(
    sensitivity_path,
    index=False
)

weight_sensitivity.to_csv(
    weight_path,
    index=False
)

threshold_sensitivity.to_csv(
    threshold_path,
    index=False
)

loo_sensitivity.to_csv(
    loo_path,
    index=False
)

# -------------------------------------------------------------------
# Figure-saving helper
# -------------------------------------------------------------------

def save_current_figure(stem):
    png_path = FIGURE_DIR / f"{stem}.png"
    pdf_path = FIGURE_DIR / f"{stem}.pdf"
    svg_path = FIGURE_DIR / f"{stem}.svg"

    plt.savefig(
        png_path,
        dpi=600,
        bbox_inches="tight"
    )

    plt.savefig(
        pdf_path,
        bbox_inches="tight"
    )

    plt.savefig(
        svg_path,
        bbox_inches="tight"
    )

    plt.close()

    return [
        png_path,
        pdf_path,
        svg_path,
    ]


created_figure_files = []

# -------------------------------------------------------------------
# Figure 1 — Ranking agreement and HIGH-tier retention
# -------------------------------------------------------------------

ranking_plot = method_full.loc[
    method_full["method"] != "Original MCI"
].copy()

x = np.arange(len(ranking_plot))
bar_width = 0.36

plt.figure(figsize=(12, 6.5))

plt.bar(
    x - bar_width / 2,
    ranking_plot["spearman_rho_vs_MCI"],
    width=bar_width,
    label="Spearman rank agreement"
)

plt.bar(
    x + bar_width / 2,
    ranking_plot["original_HIGH_recall"],
    width=bar_width,
    label="Original HIGH-tier recall"
)

plt.axhline(
    1.0,
    linewidth=0.8,
    linestyle="--"
)

plt.ylim(0, 1.08)

plt.ylabel("Agreement or recall")

plt.xlabel("Benchmark method")

plt.title(
    "MCI agreement with simpler benchmark methods"
)

plt.xticks(
    x,
    ranking_plot["method"],
    rotation=28,
    ha="right"
)

plt.legend(
    frameon=False,
    loc="lower left"
)

plt.tight_layout()

created_figure_files.extend(
    save_current_figure(
        "TASK11_FIGURE_METHOD_RANKING_AGREEMENT"
    )
)

# -------------------------------------------------------------------
# Figure 2 — Weight sensitivity
# -------------------------------------------------------------------

weight_plot = weight_sensitivity.loc[
    weight_sensitivity["scenario"]
    != "PRIMARY_40D_35S_25R"
].copy()

weight_plot["short_label"] = (
    weight_plot["scenario"]
    .str.replace(
        "_RENORMALIZED",
        "",
        regex=False
    )
    .str.replace(
        "_",
        " ",
        regex=False
    )
)

weight_plot = weight_plot.sort_values(
    "percent_tier_changes",
    ascending=True
)

plt.figure(figsize=(12, 7.5))

plt.barh(
    weight_plot["short_label"],
    weight_plot["percent_tier_changes"]
)

plt.xlabel(
    "Genes changing tier (%)"
)

plt.ylabel(
    "Weight scenario"
)

plt.title(
    "Tier sensitivity to alternative component weights"
)

plt.xlim(
    0,
    max(
        45,
        weight_plot[
            "percent_tier_changes"
        ].max() + 3
    )
)

plt.tight_layout()

created_figure_files.extend(
    save_current_figure(
        "TASK11_FIGURE_WEIGHT_SENSITIVITY"
    )
)

# -------------------------------------------------------------------
# Figure 3 — Threshold sensitivity
# -------------------------------------------------------------------

moderate_only = (
    threshold_sensitivity.loc[
        np.isclose(
            threshold_sensitivity[
                "high_threshold"
            ],
            0.70
        )
    ]
    .sort_values(
        "moderate_threshold"
    )
)

high_only = (
    threshold_sensitivity.loc[
        np.isclose(
            threshold_sensitivity[
                "moderate_threshold"
            ],
            0.45
        )
    ]
    .sort_values(
        "high_threshold"
    )
)

plt.figure(figsize=(9, 6))

plt.plot(
    moderate_only[
        "moderate_threshold"
    ],
    moderate_only[
        "percent_tier_changes"
    ],
    marker="o",
    linewidth=2,
    label="Shift MODERATE threshold; HIGH fixed at 0.70"
)

plt.plot(
    high_only[
        "high_threshold"
    ],
    high_only[
        "percent_tier_changes"
    ],
    marker="s",
    linewidth=2,
    label="Shift HIGH threshold; MODERATE fixed at 0.45"
)

plt.axvline(
    0.45,
    linewidth=0.8,
    linestyle="--"
)

plt.axvline(
    0.70,
    linewidth=0.8,
    linestyle="--"
)

plt.xlabel(
    "Threshold value"
)

plt.ylabel(
    "Genes changing tier (%)"
)

plt.title(
    "MCI tier sensitivity to decision-threshold shifts"
)

plt.legend(
    frameon=False
)

plt.tight_layout()

created_figure_files.extend(
    save_current_figure(
        "TASK11_FIGURE_THRESHOLD_SENSITIVITY"
    )
)

# -------------------------------------------------------------------
# Figure 4 — Leave-one-cohort-out robustness
# -------------------------------------------------------------------

loo_plot = loo_sensitivity.copy()

loo_plot["tier_retention"] = (
    1.0
    - loo_plot["percent_tier_changes"] / 100.0
)

x = np.arange(len(loo_plot))
bar_width = 0.36

plt.figure(figsize=(9, 6))

plt.bar(
    x - bar_width / 2,
    loo_plot[
        "spearman_rho_vs_full_MCI"
    ],
    width=bar_width,
    label="Spearman rank agreement"
)

plt.bar(
    x + bar_width / 2,
    loo_plot["tier_retention"],
    width=bar_width,
    label="Exact tier retention"
)

plt.ylim(0, 1.05)

plt.ylabel(
    "Agreement or retention"
)

plt.xlabel(
    "Cohort removed"
)

plt.title(
    "Leave-one-cohort-out MCI stability"
)

plt.xticks(
    x,
    loo_plot["held_out_cohort"]
)

plt.legend(
    frameon=False
)

plt.tight_layout()

created_figure_files.extend(
    save_current_figure(
        "TASK11_FIGURE_LEAVE_ONE_COHORT_OUT"
    )
)

# -------------------------------------------------------------------
# Figure 5 — External-evidence AUROC comparison
# -------------------------------------------------------------------

external_auroc = external.pivot(
    index="method",
    columns="external_outcome",
    values="AUROC"
).reindex(method_order)

external_auroc = external_auroc.rename(
    columns={
        "Held-out strict validation": (
            "Held-out strict"
        ),
        "Held-out directional validation": (
            "Held-out directional"
        ),
        "GWAS Catalog HCM match": (
            "GWAS match"
        ),
    }
)

x = np.arange(
    len(external_auroc.index)
)

outcome_columns = list(
    external_auroc.columns
)

bar_width = 0.24

plt.figure(figsize=(13, 7))

for index, outcome_column in enumerate(
    outcome_columns
):
    offset = (
        index
        - (len(outcome_columns) - 1) / 2
    ) * bar_width

    plt.bar(
        x + offset,
        external_auroc[
            outcome_column
        ],
        width=bar_width,
        label=outcome_column
    )

plt.axhline(
    0.5,
    linewidth=0.9,
    linestyle="--"
)

plt.ylim(0, 1.08)

plt.ylabel("AUROC")

plt.xlabel("Method")

plt.title(
    "Exploratory external-evidence discrimination\n"
    "Strict held-out results include only two positive genes"
)

plt.xticks(
    x,
    external_auroc.index,
    rotation=28,
    ha="right"
)

plt.legend(
    frameon=False,
    loc="lower left"
)

plt.tight_layout()

created_figure_files.extend(
    save_current_figure(
        "TASK11_FIGURE_EXTERNAL_EVIDENCE_AUROC"
    )
)

# -------------------------------------------------------------------
# Create a manifest of publication-ready outputs
# -------------------------------------------------------------------

all_table_files = [
    method_full_path,
    method_compact_path,
    sensitivity_path,
    weight_path,
    threshold_path,
    loo_path,
]

manifest_rows = []

for file_path in (
    all_table_files
    + created_figure_files
):
    manifest_rows.append(
        {
            "relative_path": str(
                file_path.relative_to(
                    PROJECT_DIR
                )
            ),
            "file_type": (
                file_path.suffix
                .lower()
                .replace(".", "")
            ),
            "file_size_bytes": (
                file_path.stat().st_size
            ),
        }
    )

manifest = pd.DataFrame(
    manifest_rows
)

manifest_path = (
    PUBLICATION_DIR
    / "TASK11_PUBLICATION_OUTPUT_MANIFEST.csv"
)

manifest.to_csv(
    manifest_path,
    index=False
)

# -------------------------------------------------------------------
# Print review output
# -------------------------------------------------------------------

print("=" * 125)
print("PUBLICATION-READY OUTPUTS CREATED")
print("=" * 125)

print(f"Publication directory: {PUBLICATION_DIR}")
print(f"Tables created: {len(all_table_files)}")
print(f"Figure files created: {len(created_figure_files)}")
print(
    "Each figure was exported as PNG at 600 dpi, PDF, and SVG."
)

print("\n" + "=" * 125)
print("COMPACT METHOD BENCHMARK TABLE")
print("=" * 125)

print(
    method_compact
    .round(4)
    .to_string(index=False)
)

print("\n" + "=" * 125)
print("KEY SENSITIVITY SUMMARY")
print("=" * 125)

key_weight_scenarios = [
    "EQUAL_33D_33S_33R",
    "ALT_45D_35S_20R",
    "ALT_35D_40S_25R",
    "ALT_35D_30S_35R",
    "LEAVE_OUT_R_RENORMALIZED",
    "LEAVE_OUT_S_RENORMALIZED",
    "LEAVE_OUT_D_RENORMALIZED",
]

print("\nSelected weight scenarios:")

print(
    weight_sensitivity.loc[
        weight_sensitivity[
            "scenario"
        ].isin(key_weight_scenarios),
        [
            "scenario",
            "spearman_rho_vs_primary",
            "n_tier_changes",
            "percent_tier_changes",
            "original_HIGH_recall",
        ]
    ]
    .round(4)
    .to_string(index=False)
)

print("\nLeave-one-cohort-out scenarios:")

print(
    loo_sensitivity[
        [
            "held_out_cohort",
            "spearman_rho_vs_full_MCI",
            "n_tier_changes",
            "percent_tier_changes",
            "original_HIGH_recall",
        ]
    ]
    .round(4)
    .to_string(index=False)
)

print("\n" + "=" * 125)
print("CREATED FILE MANIFEST")
print("=" * 125)

print(
    manifest.to_string(index=False)
)

print("\nManifest:")
print(manifest_path)

print("\nSTEP 10 COMPLETE")


In [ ]:
# STEP 11 — Export the current Google Colab notebook into the GitHub repository

import json
from pathlib import Path
from google.colab import _message

PROJECT_DIR = PROJECT_DIR
NOTEBOOK_DIR = PROJECT_DIR / "notebooks"
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOK_PATH = (
    NOTEBOOK_DIR
    / "TASK11_MCI_BENCHMARKING_AND_SENSITIVITY_ANALYSIS.ipynb"
)

print("=" * 100)
print("EXPORTING CURRENT COLAB NOTEBOOK")
print("=" * 100)

# Ask the active Colab session for its complete notebook content.
response = _message.blocking_request(
    "get_ipynb",
    timeout_sec=120
)

# Colab may return either the notebook directly or inside an "ipynb" key.
if isinstance(response, dict) and "ipynb" in response:
    notebook_content = response["ipynb"]
else:
    notebook_content = response

if not isinstance(notebook_content, dict):
    raise TypeError(
        "Colab did not return a valid notebook dictionary."
    )

if "cells" not in notebook_content:
    raise ValueError(
        "The returned object does not contain notebook cells."
    )

with open(
    NOTEBOOK_PATH,
    "w",
    encoding="utf-8"
) as notebook_file:
    json.dump(
        notebook_content,
        notebook_file,
        ensure_ascii=False,
        indent=1
    )

# Validate the saved notebook.
with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as notebook_file:
    saved_notebook = json.load(notebook_file)

n_total_cells = len(
    saved_notebook.get("cells", [])
)

n_code_cells = sum(
    cell.get("cell_type") == "code"
    for cell in saved_notebook.get("cells", [])
)

n_markdown_cells = sum(
    cell.get("cell_type") == "markdown"
    for cell in saved_notebook.get("cells", [])
)

file_size = NOTEBOOK_PATH.stat().st_size

print(f"Notebook path: {NOTEBOOK_PATH}")
print(f"File size: {file_size:,} bytes")
print(f"Total cells: {n_total_cells}")
print(f"Code cells: {n_code_cells}")
print(f"Markdown cells: {n_markdown_cells}")

if n_code_cells == 0:
    raise ValueError(
        "The notebook was saved, but no code cells were detected."
    )

print("\nNOTEBOOK EXPORT SUCCESSFUL")
print("STEP 11 COMPLETE")
